# Time-Series AIME (ts-AIME) APR all experiments v9.2

## One-notebook APR study using the reusable `tsaime` 0.3.1 package

This notebook contains every APR-specific component needed for the PM2.5 study:

1. Beijing, Tsukuba, and optional Thai data acquisition;
2. atmospheric preprocessing and exact-hour PM2.5 targets;
3. fixed chronological train, validation, and test periods;
4. synthetic forward/inverse operator validation;
5. strict out-of-sample S-Map prediction at 1, 6, and 24 hours;
6. pressure/wind ablation, PM2.5 negative-value sensitivity, moving-block confidence intervals, and exact structured null tests;
7. rolling vector-output ts-AIME and S-Map bridge diagnostics;
8. non-arbitrary cross-site figures, complete CSV tables, compact LaTeX tables, provenance, a manifest, and a ZIP archive.

Only reusable mathematics and generic time-series utilities are imported from `tsaime`.

No APR workflow module or external experiment source file is required.

The software is available under the PolyForm Noncommercial License 1.0.0; see `LICENSE.txt` for the controlling terms.


## Mathematical and XAI contract

For centered, column-standardized states $X\in\mathbb{R}^{n\times d}$ and forecast vectors $Y\in\mathbb{R}^{n\times q}$, vector-output ts-AIME is defined by

$$
\widehat A_\lambda
=\arg\min_A\left\{n^{-1}\lVert X-YA^\top\rVert_F^2+\lambda\lVert A\rVert_F^2\right\}
=S_{XY}(S_{YY}+\lambda I_q)^{-1}.
$$

Time-Series AIME (ts-AIME) is a regularized inverse-regression XAI operator: $A$ maps a joint forecast vector to the best regularized linear representative of the atmospheric input state under the declared loss.

The method does not assume that the forecast model is one-to-one, and $A$ is not claimed to be the exact inverse function of S-Map.

S-Map estimates the local forward response $F$ and answers how a state perturbation changes the forecast.

ts-AIME estimates the output-to-input reconstruction operator $A$ and answers which state directions are retained in the joint forecast vector.

v9.2 verifies the ridge normal equation, known synthetic forward and inverse targets, multi-horizon information gain, past-window to future-block inverse fidelity, and the linear inverse-approximation gap against a quadratic inverse diagnostic.

Pressure, humidity, precipitation, and wind results remain forecast-aligned associations, not causal effects or source-apportionment estimates.

## 1. Load `tsaime` 0.3

The notebook first uses an installed `tsaime` 0.3 package.

During local development, it can locate the surrounding `codev9` directory and install that package in editable mode.

After GitHub publication, install the released repository or wheel before running this notebook.


In [ ]:
from __future__ import annotations

import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd


def locate_tsaime_source() -> Path | None:
    explicit = os.environ.get("TSAIME_V3_ROOT") or os.environ.get("TSAIME_CODEV9_ROOT")
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents, current / "codev9", current.parent / "codev9"])
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "tsaime").is_dir():
            return candidate.resolve()
    return None


PACKAGE_ROOT = locate_tsaime_source()
try:
    installed_version = importlib.metadata.version("tsaime")
except importlib.metadata.PackageNotFoundError:
    installed_version = None

if PACKAGE_ROOT is not None:
    local_source = str(PACKAGE_ROOT / "src")
    if installed_version != "0.3.1" or local_source not in sys.path:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(PACKAGE_ROOT)]
        )
        importlib.invalidate_caches()
    if local_source not in sys.path:
        sys.path.insert(0, local_source)
elif installed_version != "0.3.1":
    raise RuntimeError(
        "tsaime 0.3.1 is required. Install the released wheel or GitHub repository, "
        "then restart the kernel."
    )

APR_REQUIREMENTS = {
    "matplotlib": "matplotlib>=3.7",
    "pyEDM": "pyEDM==2.5.7",
    "sklearn": "scikit-learn>=1.2",
}
missing_requirements = [
    requirement
    for module, requirement in APR_REQUIREMENTS.items()
    if importlib.util.find_spec(module) is None
]
if missing_requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_requirements]
    )
    importlib.invalidate_caches()

from tsaime import __version__

if __version__ != "0.3.1":
    raise RuntimeError(f"Expected tsaime 0.3.1, found {__version__}. Restart the kernel after installation.")

RESULT_BASE_DIR = Path(
    os.environ.get("TSAIME_V9_BASE_DIR", PACKAGE_ROOT or Path.cwd())
).expanduser().resolve()

print("tsaime:", __version__)
print("tsaime source:", PACKAGE_ROOT if PACKAGE_ROOT is not None else "installed package")
print("result base:", RESULT_BASE_DIR)


## 2. APR-specific atmospheric preprocessing

These functions belong to this PM2.5 paper, not to the `tsaime` package.


In [ ]:
"""Atmospheric preprocessing fixed for the APR PM2.5 v9 experiment."""

from __future__ import annotations

from typing import Sequence

import numpy as np
import pandas as pd

from tsaime.temporal import (
    ChronologicalSplit,
    complete_hourly,
    prepare_chronological_pairs,
)


DEFAULT_STATE_COLUMNS = (
    "PM25_t",
    "PM25_lag1",
    "PM25_lag24",
    "TEMP",
    "RH",
    "PRESS",
    "RAIN_EVENT",
    "LOG_RAIN",
    "WIND_U",
    "WIND_V",
)


PM25_POLICIES = ("as_reported", "negative_missing", "negative_zero")


def apply_pm25_policy(
    frame: pd.DataFrame, policy: str, *, pm25_column: str = "PM25"
) -> pd.DataFrame:
    """Apply an explicit PM2.5 negative-value policy without interpolation."""

    if policy not in PM25_POLICIES:
        raise ValueError(f"unknown PM2.5 policy: {policy}")
    data = frame.copy()
    if pm25_column not in data:
        raise KeyError(f"missing PM2.5 column: {pm25_column}")
    if policy == "negative_missing":
        data.loc[data[pm25_column] < 0, pm25_column] = np.nan
    elif policy == "negative_zero":
        data.loc[data[pm25_column] < 0, pm25_column] = 0.0
    return data


def pm25_plausibility_audit(
    frame: pd.DataFrame, *, dataset: str, site: str
) -> dict[str, object]:
    """Record PM2.5 range and negative observations before any policy is applied."""

    values = pd.to_numeric(frame["PM25"], errors="coerce")
    finite = values[np.isfinite(values)]
    temperature = pd.to_numeric(frame["TEMP"], errors="coerce")
    humidity = pd.to_numeric(frame["RH"], errors="coerce")
    pressure = pd.to_numeric(frame["PRESS"], errors="coerce")
    rain = pd.to_numeric(frame["RAIN"], errors="coerce")
    wind_speed = np.sqrt(pd.to_numeric(frame["WIND_U"], errors="coerce") ** 2 + pd.to_numeric(frame["WIND_V"], errors="coerce") ** 2)
    return {
        "dataset": dataset,
        "site": site,
        "pm25_nonmissing_n": int(finite.size),
        "pm25_negative_n": int((finite < 0).sum()),
        "pm25_zero_n": int((finite == 0).sum()),
        "pm25_min": float(finite.min()) if finite.size else np.nan,
        "pm25_max": float(finite.max()) if finite.size else np.nan,
        "temperature_min_c": float(temperature.min()),
        "temperature_max_c": float(temperature.max()),
        "rh_outside_0_100_n": int(((humidity < 0) | (humidity > 100)).sum()),
        "pressure_outside_800_1100_n": int(((pressure < 800) | (pressure > 1100)).sum()),
        "negative_rain_n": int((rain < 0).sum()),
        "wind_speed_max_m_s": float(wind_speed.max()),
        "main_policy": "as_reported",
        "sensitivity_policies": "negative_missing; negative_zero",
    }


def relative_humidity_from_dewpoint(
    temperature_c: Sequence[float] | pd.Series,
    dewpoint_c: Sequence[float] | pd.Series,
) -> np.ndarray:
    """Approximate relative humidity using the Magnus relation."""

    temperature = np.asarray(temperature_c, dtype=float)
    dewpoint = np.asarray(dewpoint_c, dtype=float)
    if temperature.shape != dewpoint.shape:
        raise ValueError("temperature and dew point must have the same shape")
    numerator = np.exp((17.625 * dewpoint) / (243.04 + dewpoint))
    denominator = np.exp((17.625 * temperature) / (243.04 + temperature))
    return np.clip(100.0 * numerator / denominator, 0.0, 100.0)


def wind_components(
    speed: Sequence[float] | pd.Series,
    direction_degrees: Sequence[float] | pd.Series,
) -> tuple[np.ndarray, np.ndarray]:
    """Convert meteorological FROM-direction and speed to east/north components."""

    speed_array = np.asarray(speed, dtype=float)
    direction = np.asarray(direction_degrees, dtype=float)
    if speed_array.shape != direction.shape:
        raise ValueError("speed and direction must have the same shape")
    theta = np.deg2rad(direction)
    return -speed_array * np.sin(theta), -speed_array * np.cos(theta)


def engineer_pm25_multihorizon_frame(
    frame: pd.DataFrame,
    horizons: Sequence[int],
    *,
    pm25_column: str = "PM25",
    date_column: str = "Date",
) -> tuple[pd.DataFrame, list[str], list[str]]:
    """Create the APR-specific PM2.5 state and target columns on an hourly grid."""

    horizon_values = tuple(int(value) for value in horizons)
    if not horizon_values or any(value < 1 for value in horizon_values):
        raise ValueError("horizons must contain positive integers")
    if len(set(horizon_values)) != len(horizon_values):
        raise ValueError("horizons must not contain duplicates")
    data = complete_hourly(frame, date_column=date_column).set_index(date_column).sort_index()
    if pm25_column not in data:
        raise KeyError(f"missing PM2.5 column: {pm25_column}")
    data["PM25_t"] = data[pm25_column]
    data["PM25_lag1"] = data[pm25_column].shift(1)
    data["PM25_lag24"] = data[pm25_column].shift(24)
    if "RAIN" in data:
        data["RAIN_EVENT"] = np.where(
            data["RAIN"].notna(), (data["RAIN"] > 0).astype(float), np.nan
        )
        data["LOG_RAIN"] = np.log1p(data["RAIN"].clip(lower=0))
    targets = []
    for horizon in horizon_values:
        target = f"PM25_future_{horizon}h"
        data[target] = data[pm25_column].shift(-horizon)
        targets.append(target)
    states = [column for column in DEFAULT_STATE_COLUMNS if column in data]
    return data.reset_index(), states, targets


def prepare_pm25_multihorizon_pairs(
    frame: pd.DataFrame,
    horizons: Sequence[int],
    split: ChronologicalSplit,
    *,
    origin_step_hours: int = 1,
    required_states: Sequence[str] | None = None,
    date_column: str = "Date",
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str]]:
    """Build exact-time PM2.5 pairs and apply the generic chronological split."""

    horizon_values = tuple(int(value) for value in horizons)
    engineered, states, targets = engineer_pm25_multihorizon_frame(
        frame,
        horizon_values,
        date_column=date_column,
    )
    if required_states is not None:
        states = [str(value) for value in required_states]
        missing = [column for column in states if column not in engineered]
        if missing:
            raise KeyError(f"missing required state columns: {missing}")
    candidates, complete = prepare_chronological_pairs(
        engineered,
        states,
        targets,
        split,
        max_target_horizon_hours=max(horizon_values),
        origin_step_hours=origin_step_hours,
        date_column=date_column,
    )
    return candidates, complete, states, targets


## 3. APR data acquisition and harmonization

The URLs, station formats, meteorological conversions, and Thai-site metadata are fixed here so that the notebook records its own data contract.


In [ ]:
"""APR-specific public data loaders; this module is not part of ``tsaime``."""

from __future__ import annotations

import gzip
import hashlib
import io
import shutil
import urllib.parse
import urllib.request
import zipfile
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from pathlib import Path
from typing import Mapping, Sequence

import numpy as np
import pandas as pd

from tsaime.temporal import complete_hourly



UCI_BEIJING_URL = "https://archive.ics.uci.edu/static/public/501/beijing%2Bmulti%2Bsite%2Bair%2Bquality%2Bdata.zip"
UCI_BEIJING_LANDING_PAGE = "https://archive.ics.uci.edu/dataset/501/beijingmultisiteairqualitydata"
UCI_BEIJING_DOI = "10.24432/C5RK5G"
NIES_LANDING_PAGE = "https://db.cger.nies.go.jp/MD/10.17595/20250418.001.html.en"
NIES_DOI = "10.17595/20250418.001"
NIES_VERSION = "1.0"
NIES_TEMPLATE = "https://db.cger.nies.go.jp/nies_data/10.17595/20250418.001/AMEL.hourly.{year}.Ver1.0.txt"
SOURCE_ACCESSED_ON = "2026-09-04"
OPENAQ_BUCKET = "https://openaq-data-archive.s3.amazonaws.com/"
NASA_POWER_ENDPOINT = "https://power.larc.nasa.gov/api/temporal/hourly/point"

THAI_SITES: dict[str, dict[str, float | int]] = {
    "Chonburi": {"location_id": 225654, "latitude": 13.35461667, "longitude": 100.9792167},
    "Chiang_Mai": {"location_id": 225669, "latitude": 18.840732, "longitude": 98.96978},
}

COMPASS_DEGREES = {
    "N": 0.0, "NNE": 22.5, "NE": 45.0, "ENE": 67.5,
    "E": 90.0, "ESE": 112.5, "SE": 135.0, "SSE": 157.5,
    "S": 180.0, "SSW": 202.5, "SW": 225.0, "WSW": 247.5,
    "W": 270.0, "WNW": 292.5, "NW": 315.0, "NNW": 337.5,
}


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


@dataclass
class PublicDataRepository:
    """Download and cache the public datasets used by the reference workflow."""

    data_dir: Path | str
    user_agent: str = "tsAIME-v9-research/1.0"
    records: list[dict[str, object]] = field(default_factory=list)
    license_records: list[dict[str, object]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.data_dir = Path(self.data_dir).expanduser().resolve()
        self.data_dir.mkdir(parents=True, exist_ok=True)

    def download(self, url: str, relative_path: str | Path, timeout: int = 180) -> Path:
        destination = self.data_dir / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        cached = destination.exists() and destination.stat().st_size > 0
        if not cached:
            request = urllib.request.Request(url, headers={"User-Agent": self.user_agent})
            temporary = destination.with_suffix(destination.suffix + ".part")
            with urllib.request.urlopen(request, timeout=timeout) as response, temporary.open("wb") as handle:
                shutil.copyfileobj(response, handle)
            temporary.replace(destination)
        self.records.append(
            {
                "url": url,
                "path": str(destination.relative_to(self.data_dir)),
                "bytes": destination.stat().st_size,
                "sha256": _sha256(destination),
                "cached": cached,
                "file_mtime_utc": datetime.fromtimestamp(
                    destination.stat().st_mtime, tz=timezone.utc
                ).isoformat(),
                "verified_in_run_utc": datetime.now(timezone.utc).isoformat(),
            }
        )
        return destination

    def load_beijing_sites(self, limit: int = 12) -> dict[str, pd.DataFrame]:
        """Load UCI Beijing Multi-Site data, including the current nested ZIP."""

        archive = self.download(UCI_BEIJING_URL, "uci_beijing_multisite.zip")
        payloads: list[tuple[str, bytes]] = []
        with zipfile.ZipFile(archive) as outer:
            nested = sorted(name for name in outer.namelist() if name.lower().endswith(".zip"))
            if nested:
                with zipfile.ZipFile(io.BytesIO(outer.read(nested[0]))) as source:
                    members = sorted(
                        name for name in source.namelist()
                        if name.lower().endswith(".csv") and not Path(name).name.startswith("._")
                    )
                    payloads = [(name, source.read(name)) for name in members[:limit]]
            else:
                members = sorted(
                    name for name in outer.namelist()
                    if name.lower().endswith(".csv") and not Path(name).name.startswith("._")
                )
                payloads = [(name, outer.read(name)) for name in members]
        required = {"year", "month", "day", "hour", "PM2.5", "TEMP", "DEWP", "PRES", "RAIN", "wd", "WSPM"}
        sites: dict[str, pd.DataFrame] = {}
        for member, payload in payloads:
            raw = pd.read_csv(io.BytesIO(payload))
            if not required.issubset(raw.columns):
                continue
            raw["Date"] = pd.to_datetime(
                dict(year=raw["year"], month=raw["month"], day=raw["day"], hour=raw["hour"])
            )
            site = (
                str(raw["station"].dropna().iloc[0])
                if "station" in raw and raw["station"].notna().any()
                else Path(member).stem
            )
            degrees = raw["wd"].astype(str).str.upper().map(COMPASS_DEGREES)
            u, v = wind_components(raw["WSPM"], degrees)
            frame = pd.DataFrame(
                {
                    "Date": raw["Date"],
                    "PM25": pd.to_numeric(raw["PM2.5"], errors="coerce"),
                    "TEMP": pd.to_numeric(raw["TEMP"], errors="coerce"),
                    "RH": relative_humidity_from_dewpoint(raw["TEMP"], raw["DEWP"]),
                    "PRESS": pd.to_numeric(raw["PRES"], errors="coerce"),
                    "RAIN": pd.to_numeric(raw["RAIN"], errors="coerce"),
                    "WIND_U": u,
                    "WIND_V": v,
                }
            )
            sites[site] = complete_hourly(frame)
        if not sites:
            raise RuntimeError("no valid Beijing station CSV was found")
        return sites

    def load_tsukuba(self, years: Sequence[int] = tuple(range(2017, 2023))) -> pd.DataFrame:
        frames = []
        for year in years:
            path = self.download(
                NIES_TEMPLATE.format(year=int(year)),
                f"AMEL.hourly.{int(year)}.Ver1.0.txt",
            )
            header = path.read_text(encoding="utf-8", errors="replace").splitlines()[:100]
            embedded = next((line.strip(" -") for line in header if "CC BY" in line), "not found")
            self.license_records.append(
                {
                    "dataset": "Tsukuba", "file": path.name,
                    "landing_page_license": "CC BY-NC-ND 4.0",
                    "embedded_file_notice": embedded,
                    "accessed_on": SOURCE_ACCESSED_ON,
                    "status": "conflict: verify controlling publisher terms before release",
                }
            )
            frames.append(_parse_nies_year(path))
        return complete_hourly(pd.concat(frames, ignore_index=True))

    def load_thai_site(
        self,
        name: str,
        metadata: Mapping[str, float | int],
        years: Sequence[int],
        *,
        workers: int = 4,
    ) -> pd.DataFrame:
        pm25 = self._load_openaq_pm25(int(metadata["location_id"]), years, workers=workers)
        meteorology = self._load_nasa_power(
            float(metadata["latitude"]),
            float(metadata["longitude"]),
            f"{min(years)}0101",
            f"{max(years)}1231",
            name,
        )
        merged = pm25.merge(meteorology, on="DateUTC", how="outer").sort_values("DateUTC")
        merged["Date"] = merged["DateUTC"].dt.tz_convert("Asia/Bangkok").dt.tz_localize(None)
        return complete_hourly(merged.drop(columns="DateUTC"))

    def _openaq_keys(self, location_id: int, years: Sequence[int]) -> list[str]:
        prefix = f"records/csv.gz/locationid={location_id}/"
        token: str | None = None
        keys: list[str] = []
        while True:
            params = {"list-type": "2", "prefix": prefix}
            if token:
                params["continuation-token"] = token
            request = urllib.request.Request(
                OPENAQ_BUCKET + "?" + urllib.parse.urlencode(params),
                headers={"User-Agent": self.user_agent},
            )
            with urllib.request.urlopen(request, timeout=180) as response:
                root = ET.fromstring(response.read())
            namespace = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
            for node in root.findall("s3:Contents/s3:Key", namespace):
                key = node.text or ""
                if any(f"/year={int(year)}/" in key for year in years):
                    keys.append(key)
            truncated = root.findtext(
                "s3:IsTruncated", default="false", namespaces=namespace
            ).lower() == "true"
            if not truncated:
                break
            token = root.findtext(
                "s3:NextContinuationToken", default="", namespaces=namespace
            )
            if not token:
                break
        return sorted(keys)

    def _load_openaq_pm25(
        self, location_id: int, years: Sequence[int], *, workers: int
    ) -> pd.DataFrame:
        keys = self._openaq_keys(location_id, years)
        if not keys:
            raise RuntimeError(f"no OpenAQ objects found for location {location_id}")

        def fetch(key: str) -> Path:
            relative = Path(f"openaq_{location_id}") / key.replace("/", "__")
            return self.download(
                OPENAQ_BUCKET + urllib.parse.quote(key, safe="/="), relative, timeout=120
            )

        paths: list[Path] = []
        with ThreadPoolExecutor(max_workers=max(1, workers)) as executor:
            futures = [executor.submit(fetch, key) for key in keys]
            for future in as_completed(futures):
                try:
                    paths.append(future.result())
                except Exception:
                    continue
        rows = []
        for path in sorted(paths):
            try:
                daily = pd.read_csv(path, compression="gzip")
                parameter = "parameter" if "parameter" in daily else next(
                    column for column in daily if "parameter" in column.lower()
                )
                date = "datetime" if "datetime" in daily else next(
                    column for column in daily if "date" in column.lower()
                )
                value = "value" if "value" in daily else next(
                    column for column in daily if column.lower() == "value"
                )
                normalized = daily[parameter].astype(str).str.lower().str.replace(".", "", regex=False)
                selected = daily.loc[normalized.isin({"pm25", "pm₂5"}), [date, value]].copy()
                selected.columns = ["DateUTC", "PM25"]
                rows.append(selected)
            except Exception:
                continue
        if not rows:
            raise RuntimeError(f"OpenAQ objects for {location_id} contained no PM2.5 rows")
        data = pd.concat(rows, ignore_index=True)
        data["DateUTC"] = pd.to_datetime(data["DateUTC"], utc=True, errors="coerce").dt.floor("h")
        data["PM25"] = pd.to_numeric(data["PM25"], errors="coerce")
        data.loc[data["PM25"] < 0, "PM25"] = np.nan
        return data.groupby("DateUTC", as_index=False)["PM25"].mean()

    def _load_nasa_power(
        self, latitude: float, longitude: float, start: str, end: str, name: str
    ) -> pd.DataFrame:
        query = urllib.parse.urlencode(
            {
                "parameters": "T2M,RH2M,PS,PRECTOT,U10M,V10M",
                "community": "SB",
                "longitude": longitude,
                "latitude": latitude,
                "start": start,
                "end": end,
                "format": "CSV",
                "time-standard": "UTC",
            }
        )
        safe_name = "".join(character if character.isalnum() else "_" for character in name)
        path = self.download(
            f"{NASA_POWER_ENDPOINT}?{query}",
            f"nasa_power_{safe_name}_{start}_{end}.csv",
        )
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        header_end = next(index for index, line in enumerate(lines) if "-END HEADER-" in line)
        data = pd.read_csv(io.StringIO("\n".join(lines[header_end + 1 :])))
        data["DateUTC"] = pd.to_datetime(
            dict(year=data["YEAR"], month=data["MO"], day=data["DY"], hour=data["HR"]),
            utc=True,
        )
        for column in ["T2M", "RH2M", "PS", "PRECTOT", "U10M", "V10M"]:
            data[column] = pd.to_numeric(data[column], errors="coerce").mask(
                lambda values: values <= -900
            )
        data["PS"] = 10.0 * data["PS"]  # NASA POWER kPa to common hPa
        return data.rename(
            columns={
                "T2M": "TEMP",
                "RH2M": "RH",
                "PS": "PRESS",
                "PRECTOT": "RAIN",
                "U10M": "WIND_U",
                "V10M": "WIND_V",
            }
        )[["DateUTC", "TEMP", "RH", "PRESS", "RAIN", "WIND_U", "WIND_V"]]


NIES_COLUMNS = [
    "TIME", "YEAR", "MONTH", "DAY", "HOUR", "CH4", "NMHC", "O3", "SO2", "NO",
    "NO2", "SPM", "PM25", "TEMP", "RH", "PRESS", "SR", "UVA", "RAIN", "WD", "WS",
]
NIES_SENTINELS = {
    "CH4": 9.99, "NMHC": 9.99, "O3": 999, "SO2": 999, "NO": 999, "NO2": 999,
    "SPM": 999, "PM25": 9999, "TEMP": 99.9, "RH": 999, "PRESS": 9999,
    "SR": 9.99, "UVA": 999, "RAIN": 999.9, "WD": 99, "WS": 99.9,
}


def _parse_nies_year(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        header_lines = int(handle.readline().split()[0])
    raw = pd.read_csv(
        path,
        sep=r"\s+",
        skiprows=header_lines,
        names=NIES_COLUMNS,
        engine="python",
        on_bad_lines="skip",
    )
    for column in NIES_COLUMNS:
        raw[column] = pd.to_numeric(raw[column], errors="coerce")
    for column, sentinel in NIES_SENTINELS.items():
        raw.loc[np.isclose(raw[column], sentinel, equal_nan=False), column] = np.nan
    base = pd.to_datetime(
        dict(year=raw["YEAR"], month=raw["MONTH"], day=raw["DAY"]), errors="coerce"
    )
    raw["Date"] = base + pd.to_timedelta(raw["HOUR"], unit="h")
    code = raw["WD"]
    degrees = ((code % 16.0) * 22.5).where(code.between(1, 16))
    u, v = wind_components(raw["WS"], degrees)
    calm = (code == 0) | (raw["WS"] < 0.4)
    u = pd.Series(u).mask(calm, 0.0).to_numpy()
    v = pd.Series(v).mask(calm, 0.0).to_numpy()
    return pd.DataFrame(
        {
            "Date": raw["Date"],
            "PM25": raw["PM25"],
            "TEMP": raw["TEMP"],
            "RH": raw["RH"],
            "PRESS": raw["PRESS"],
            "RAIN": raw["RAIN"],
            "WIND_U": u,
            "WIND_V": v,
        }
    ).dropna(subset=["Date"])


def source_table() -> pd.DataFrame:
    """Return the prespecified data-source roles for reporting."""

    return pd.DataFrame(
        [
            {
                "dataset": "Beijing",
                "provider": "UCI / Beijing Municipal Environmental Monitoring Center",
                "period_used": "2013-03-01 to 2017-02-28",
                "resolution": "hourly",
                "sites_planned": 12,
                "doi": UCI_BEIJING_DOI,
                "version": "UCI dataset 501 (download snapshot)",
                "accessed_on": SOURCE_ACCESSED_ON,
                "landing_page": UCI_BEIJING_LANDING_PAGE,
                "license_or_terms": "CC BY 4.0",
                "redistribution": "raw data excluded from result ZIP",
                "role": "required multi-site analysis",
            },
            {
                "dataset": "Tsukuba",
                "provider": "National Institute for Environmental Studies (NIES)",
                "period_used": "2017-01-01 to 2022-12-31",
                "resolution": "hourly",
                "sites_planned": 1,
                "doi": NIES_DOI,
                "version": NIES_VERSION,
                "accessed_on": SOURCE_ACCESSED_ON,
                "landing_page": NIES_LANDING_PAGE,
                "license_or_terms": "landing page: CC BY-NC-ND 4.0; downloaded text header observed as CC BY 4.0; recheck current publisher terms before release",
                "redistribution": "no raw redistribution",
                "role": "required external-region validation",
            },
            {
                "dataset": "Thailand",
                "provider": "Air4Thai via OpenAQ + NASA POWER",
                "period_used": "2021-2025 when available",
                "resolution": "hourly",
                "sites_planned": 2,
                "doi": "not assigned for combined optional input",
                "version": "source snapshot at run time",
                "accessed_on": "record when extension is enabled",
                "landing_page": "https://openaq.org/; https://power.larc.nasa.gov/",
                "license_or_terms": "source terms must be verified when enabled",
                "redistribution": "no raw redistribution",
                "role": "optional quality-gated extension",
            },
        ]
    )


## 4. APR configuration and study periods

The following cell defines paper-specific states, horizons, chronological boundaries, quality rules, and result objects.


In [ ]:
"""APR-specific PM2.5 workflow built on the reusable ``tsaime`` API."""

from __future__ import annotations

import argparse
import json
import os
import platform
import sys
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence

import numpy as np
import pandas as pd

from tsaime.artifacts import ArtifactWriter
from tsaime.operators import (
    MatrixStandardizer,
    covariance_weighted_inverse,
    fit_inverse_operator,
    marginal_cross_correlation,
    operator_cosine,
    scalar_aime_operator,
)
from tsaime.rolling import RollingVectorTSAIME, RollingVectorTSAIMEConfig
from tsaime.smap import coefficients_in_window_coordinates, run_smap_oos, select_smap_theta
from tsaime.statistics import (
    moving_block_skill_interval,
    regression_metrics,
    structured_period_shift_test,
)
from tsaime.temporal import (
    ChronologicalSplit,
    QualityGate,
    audit_hourly_site,
    evaluate_quality_gate,
)



STATE_COLUMNS = (
    "PM25_t",
    "PM25_lag1",
    "PM25_lag24",
    "TEMP",
    "RH",
    "PRESS",
    "RAIN_EVENT",
    "LOG_RAIN",
    "WIND_U",
    "WIND_V",
)


@dataclass(frozen=True)
class APRExperimentConfig:
    """Complete configuration for one reproducible APR workflow run."""

    run_mode: str = "publication"
    enable_thailand: bool = False
    main_pm25_policy: str = "as_reported"
    seed: int = 20260904
    horizons: tuple[int, ...] = (1, 6, 24)
    origin_step_hours: int | None = None
    validation_stride: int | None = None
    theta_grid: tuple[float, ...] | None = None
    inverse_ridge_grid: tuple[float, ...] = (0.0, 1e-3, 1e-2, 1e-1, 1.0)
    forecast_ridge_grid: tuple[float, ...] = (0.0, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1000.0, 10000.0)
    quadratic_ridge_grid: tuple[float, ...] = (0.0, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1000.0)
    aime_window_days: int = 28
    aime_window_sensitivity_days: tuple[int, ...] = (14, 28, 56)
    aime_endpoint_step_days: int | None = None
    bootstrap_resamples: int | None = None
    null_permutations: int | None = None
    beijing_site_limit: int | None = None
    synthetic_rows_per_regime: int | None = None
    synthetic_regimes: int | None = None
    synthetic_tracking_replicates: int | None = None
    clear_outputs: bool = True

    def resolved(self) -> "APRExperimentConfig":
        if self.run_mode not in {"publication", "smoke"}:
            raise ValueError("run_mode must be 'publication' or 'smoke'")
        if self.main_pm25_policy != "as_reported":
            raise ValueError("the prespecified main analysis must use as_reported PM2.5")
        publication = self.run_mode == "publication"
        return APRExperimentConfig(
            **{
                **asdict(self),
                "origin_step_hours": self.origin_step_hours or (6 if publication else 24),
                "validation_stride": self.validation_stride or (4 if publication else 2),
                "theta_grid": self.theta_grid or ((0.0, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0) if publication else (0.0, 2.0, 8.0)),
                "aime_endpoint_step_days": self.aime_endpoint_step_days or (7 if publication else 28),
                "bootstrap_resamples": self.bootstrap_resamples or (999 if publication else 99),
                "null_permutations": self.null_permutations,
                "beijing_site_limit": self.beijing_site_limit or (12 if publication else 2),
                "synthetic_rows_per_regime": self.synthetic_rows_per_regime or (600 if publication else 180),
                "synthetic_regimes": self.synthetic_regimes or (20 if publication else 4),
                "synthetic_tracking_replicates": self.synthetic_tracking_replicates or (20 if publication else 4),
            }
        )


@dataclass(frozen=True)
class APRRunResult:
    """Paths and final status returned to notebooks and command-line callers."""

    run_id: str
    run_root: Path
    output_dir: Path
    zip_path: Path
    readiness: pd.DataFrame
    quality_gate: pd.DataFrame
    forecast_skill: pd.DataFrame


SPLITS = {
    "Beijing": ChronologicalSplit("2013-03-01", "2015-03-01", "2016-03-01", "2017-03-01"),
    "Tsukuba": ChronologicalSplit("2017-01-01", "2020-01-01", "2021-01-01", "2023-01-01"),
    "Thailand": ChronologicalSplit("2021-04-29", "2024-01-01", "2025-01-01", "2026-01-01"),
}


## 5. Synthetic validation and strict OOS site analysis

These functions implement the APR experiments by calling the reusable `tsaime` estimators.


In [ ]:
def _regime_season(timestamp: pd.Timestamp) -> str:
    month = timestamp.month
    if month in (12, 1, 2):
        return "DJF"
    if month in (3, 4, 5):
        return "MAM"
    if month in (6, 7, 8):
        return "JJA"
    return "SON"


def _random_correlation(dimension: int, generator: np.random.Generator) -> np.ndarray:
    raw = generator.normal(size=(dimension, dimension))
    covariance = raw @ raw.T
    scale = np.sqrt(np.diag(covariance))
    return covariance / np.outer(scale, scale)


def _synthetic_time_local_tracking(config: APRExperimentConfig) -> pd.DataFrame:
    """Repeat known-regime recovery to quantify time-local sampling variation."""

    d, q = len(STATE_COLUMNS), len(config.horizons)
    pressure_index = STATE_COLUMNS.index("PRESS")
    wind_index = STATE_COLUMNS.index("WIND_U")
    changes = (-0.55, -0.15, 0.25, 0.65)
    rows: list[dict[str, float | int]] = []
    for replicate in range(int(config.synthetic_tracking_replicates)):
        replicate_seed = config.seed + 90000 + replicate
        generator = np.random.default_rng(replicate_seed)
        sigma = 0.7 * _random_correlation(d, generator) + 0.3 * np.eye(d)
        base_forward = generator.normal(scale=0.18, size=(q, d))
        base_forward[:, :3] += np.asarray(
            [[0.8, 0.25, -0.1], [0.55, 0.4, 0.15], [0.25, 0.35, 0.45]]
        )[:q]
        for segment, change in enumerate(changes, start=1):
            forward = base_forward.copy()
            forward[:, pressure_index] += change
            forward[:, wind_index] -= 0.7 * change
            n = max(240, int(config.synthetic_rows_per_regime))
            x = generator.multivariate_normal(np.zeros(d), sigma, size=n)
            noise_sd = 0.08
            y = x @ forward.T + generator.normal(scale=noise_sd, size=(n, q))
            model = fit_inverse_operator(x, y, ridge=0.01)
            x_z = model.x_scaler.transform(x)
            forward_z = (
                np.diag(1.0 / model.y_scaler.scale)
                @ forward
                @ np.diag(model.x_scaler.scale)
            )
            sigma_z = np.cov(x_z, rowvar=False, ddof=0)
            noise_cov_z = np.diag((noise_sd / model.y_scaler.scale) ** 2)
            truth = covariance_weighted_inverse(
                forward_z, sigma_z, ridge=0.01,
                output_noise_covariance=noise_cov_z,
            )
            rows.append(
                {
                    "replicate": replicate + 1,
                    "replicate_seed": replicate_seed,
                    "time_segment": segment,
                    "forward_pressure_change": change,
                    "AIME_truth_pressure_1h": truth[pressure_index, 0],
                    "AIME_estimated_pressure_1h": model.operator[pressure_index, 0],
                    "AIME_truth_wind_u_1h": truth[wind_index, 0],
                    "AIME_estimated_wind_u_1h": model.operator[wind_index, 0],
                    "inverse_relative_error": float(
                        np.linalg.norm(model.operator - truth) / np.linalg.norm(truth)
                    ),
                    "inverse_operator_cosine": operator_cosine(model.operator, truth),
                }
            )
    return pd.DataFrame(rows)


def _ridge_inverse_objective_diagnostic(
    model: Any,
    inputs: np.ndarray,
    outputs: np.ndarray,
    marginal: np.ndarray,
) -> dict[str, float | bool]:
    """Check the declared ridge objective and its normal equation numerically."""

    x_z = model.x_scaler.transform(inputs)
    y_z = model.y_scaler.transform(outputs)
    regularized_covariance = model.output_covariance + model.ridge * np.eye(y_z.shape[1])
    residual = model.operator @ regularized_covariance - model.cross_covariance

    def objective(operator: np.ndarray) -> float:
        reconstruction_error = x_z - y_z @ operator.T
        return float(
            np.mean(np.sum(reconstruction_error**2, axis=1))
            + model.ridge * np.sum(operator**2)
        )

    optimum = objective(model.operator)
    marginal_value = objective(marginal)
    zero_value = objective(np.zeros_like(model.operator))
    return {
        "normal_equation_relative_residual": float(
            np.linalg.norm(residual) / max(np.linalg.norm(model.cross_covariance), 1e-12)
        ),
        "ridge_objective_tsAIME": optimum,
        "ridge_objective_marginal": marginal_value,
        "ridge_objective_zero": zero_value,
        "tsAIME_beats_marginal_objective": bool(optimum <= marginal_value + 1e-12),
        "tsAIME_beats_zero_objective": bool(optimum <= zero_value + 1e-12),
    }


def _synthetic_experiment(
    config: APRExperimentConfig, writer: ArtifactWriter
) -> dict[str, Any]:
    generator = np.random.default_rng(config.seed)
    d, q = len(STATE_COLUMNS), len(config.horizons)
    scalar_x = generator.normal(size=(1000, 7))
    scalar_y = 0.8 * scalar_x[:, 0] - 0.4 * scalar_x[:, 2] + generator.normal(scale=0.3, size=1000)
    scalar, _ = scalar_aime_operator(scalar_x, scalar_y)
    correlations = np.asarray([
        np.corrcoef(scalar_x[:, index], scalar_y)[0, 1]
        for index in range(scalar_x.shape[1])
    ])
    scalar_table = pd.DataFrame(
        {
            "feature": [f"x{index + 1}" for index in range(len(scalar))],
            "scalar_AIME": scalar,
            "Pearson_r": correlations,
            "absolute_difference": np.abs(scalar - correlations),
        }
    )
    scalar_max_difference = float(scalar_table["absolute_difference"].max())
    writer.save_table(
        scalar_table,
        "table02_scalar_equivalence",
        "Scalar-output standardized AIME equals Pearson correlation.",
        "tab:scalar_equivalence",
    )

    metrics: list[dict[str, float | int]] = []
    tasks: list[dict[str, float | int | str]] = []
    objective_rows: list[dict[str, float | int | bool]] = []
    for regime in range(int(config.synthetic_regimes)):
        replicate_seed = config.seed + 1000 + regime
        replicate_generator = np.random.default_rng(replicate_seed)
        n = int(config.synthetic_rows_per_regime)
        sigma = 0.75 * _random_correlation(d, replicate_generator) + 0.25 * np.eye(d)
        forward = replicate_generator.normal(scale=0.28, size=(q, d))
        forward[:, :3] += np.asarray(
            [[0.8, 0.25, -0.1], [0.55, 0.4, 0.15], [0.25, 0.35, 0.45]]
        )[:q]
        x = replicate_generator.multivariate_normal(np.zeros(d), sigma, size=n)
        y_signal = x @ forward.T
        noise_sd = 0.06
        y = y_signal + replicate_generator.normal(scale=noise_sd, size=y_signal.shape)
        cut = int(n * 0.65)
        dates = pd.date_range("2000-01-01", periods=n, freq="h")
        table = pd.DataFrame(x, columns=STATE_COLUMNS)
        targets = [f"synthetic_y_{h}" for h in config.horizons]
        for index, target in enumerate(targets):
            table[target] = y[:, index]
        table["Date"] = dates
        library = table.iloc[:cut].reset_index(drop=True)
        prediction = table.iloc[cut:].reset_index(drop=True)

        smap = run_smap_oos(
            library,
            prediction,
            STATE_COLUMNS,
            targets,
            theta=2.0,
        )
        forward_hat = np.nanmedian(smap.coefficients_standardized, axis=0)
        inverse_model = fit_inverse_operator(x[:cut], y[:cut], ridge=0.01)
        x_z = inverse_model.x_scaler.transform(x)
        y_z = inverse_model.y_scaler.transform(y)
        forward_z = (
            np.diag(1.0 / inverse_model.y_scaler.scale)
            @ forward
            @ np.diag(inverse_model.x_scaler.scale)
        )
        sigma_z = np.cov(x_z[:cut], rowvar=False, ddof=0)
        noise_cov_z = np.diag((noise_sd / inverse_model.y_scaler.scale) ** 2)
        inverse_truth = covariance_weighted_inverse(
            forward_z, sigma_z, ridge=0.01, output_noise_covariance=noise_cov_z
        )
        marginal = marginal_cross_correlation(x[:cut], y[:cut])
        f_error = float(np.linalg.norm(forward_hat - forward_z) / np.linalg.norm(forward_z))
        a_error = float(
            np.linalg.norm(inverse_model.operator - inverse_truth) / np.linalg.norm(inverse_truth)
        )
        marginal_error = float(np.linalg.norm(marginal - inverse_truth) / np.linalg.norm(inverse_truth))
        objective_rows.append({
            "regime": regime + 1,
            "replicate_seed": replicate_seed,
            **_ridge_inverse_objective_diagnostic(
                inverse_model, x[:cut], y[:cut], marginal
            ),
        })
        closure = float(
            np.linalg.norm(forward_z @ inverse_model.operator - np.eye(q)) / np.linalg.norm(np.eye(q))
        )
        projection = inverse_model.operator @ forward_z
        idempotence = float(
            np.linalg.norm(projection @ projection - projection)
            / max(np.linalg.norm(projection), 1e-12)
        )
        metrics.append(
            {
                "regime": regime + 1,
                "replicate_seed": replicate_seed,
                "forward_relative_error_SMap": f_error,
                "inverse_relative_error_vector_AIME": a_error,
                "inverse_relative_error_marginal_correlations": marginal_error,
                "forward_inverse_closure_error": closure,
                "input_projection_idempotence_error": idempotence,
                "operator_cosine_AIME_truth": operator_cosine(inverse_model.operator, inverse_truth),
            }
        )
        x_test_z = x_z[cut:]
        y_test_z = y_z[cut:]
        task_values = {
            ("forward_Y_from_X", "SMap_forward"): x_test_z @ forward_hat.T,
            ("forward_Y_from_X", "AIME_transpose_forward"): x_test_z @ inverse_model.operator,
            ("inverse_X_from_Y", "AIME_inverse"): y_test_z @ inverse_model.operator.T,
            ("inverse_X_from_Y", "SMap_transpose_inverse"): y_test_z @ forward_hat,
            ("inverse_X_from_Y", "marginal_correlation_inverse"): y_test_z @ marginal.T,
        }
        for (task, method), values in task_values.items():
            truth = y_test_z if task == "forward_Y_from_X" else x_test_z
            tasks.append(
                {
                    "regime": regime + 1,
                    "replicate_seed": replicate_seed,
                    "task": task,
                    "method": method,
                    "RMSE": float(np.sqrt(np.mean((values - truth) ** 2))),
                }
            )

    metric_table = pd.DataFrame(metrics)
    task_table = pd.DataFrame(tasks)
    objective_table = pd.DataFrame(objective_rows)
    tracking_table = _synthetic_time_local_tracking(config)
    tracking_summary = tracking_table.groupby(
        ["time_segment", "forward_pressure_change"], as_index=False
    ).agg(
        replicates=("replicate", "nunique"),
        A_press_true=("AIME_truth_pressure_1h", "median"),
        A_press_est=("AIME_estimated_pressure_1h", "median"),
        A_wind_true=("AIME_truth_wind_u_1h", "median"),
        A_wind_est=("AIME_estimated_wind_u_1h", "median"),
        median_rel_error=("inverse_relative_error", "median"),
        q95_rel_error=("inverse_relative_error", lambda values: values.quantile(0.95)),
        median_cosine=("inverse_operator_cosine", "median"),
    )
    recovery_latex = pd.DataFrame(
        [
            {"estimand": "forward F", "method": "S-Map", "median_error": metric_table["forward_relative_error_SMap"].median(), "q25": metric_table["forward_relative_error_SMap"].quantile(0.25), "q75": metric_table["forward_relative_error_SMap"].quantile(0.75)},
            {"estimand": "inverse A", "method": "ts-AIME", "median_error": metric_table["inverse_relative_error_vector_AIME"].median(), "q25": metric_table["inverse_relative_error_vector_AIME"].quantile(0.25), "q75": metric_table["inverse_relative_error_vector_AIME"].quantile(0.75)},
            {"estimand": "inverse A", "method": "marginal r", "median_error": metric_table["inverse_relative_error_marginal_correlations"].median(), "q25": metric_table["inverse_relative_error_marginal_correlations"].quantile(0.25), "q75": metric_table["inverse_relative_error_marginal_correlations"].quantile(0.75)},
        ]
    )
    writer.save_table(
        metric_table,
        "table03_synthetic_operator_recovery",
        "Synthetic recovery of forward and inverse operators.",
        "tab:synthetic_operator",
        latex_frame=recovery_latex,
    )
    writer.save_table(
        task_table,
        "table04_synthetic_cross_task",
        "Forward and inverse cross-task errors in the synthetic experiment.",
        "tab:synthetic_cross_task",
        latex_frame=task_table.groupby(["task", "method"], as_index=False).agg(
            replicates=("regime", "nunique"),
            median_RMSE=("RMSE", "median"),
            q25_RMSE=("RMSE", lambda values: values.quantile(0.25)),
            q75_RMSE=("RMSE", lambda values: values.quantile(0.75)),
        ),
    )
    writer.save_table(
        objective_table,
        "table03b_inverse_objective_validation",
        "Numerical validation of the declared regularized inverse-regression objective.",
        "tab:inverse_objective",
        latex_frame=pd.DataFrame([{
            "replicates": int(len(objective_table)),
            "max_normal_residual": objective_table["normal_equation_relative_residual"].max(),
            "AIME_beats_marginal": int(objective_table["tsAIME_beats_marginal_objective"].sum()),
            "AIME_beats_zero": int(objective_table["tsAIME_beats_zero_objective"].sum()),
        }]),
    )
    writer.save_table(
        tracking_table,
        "table04a_synthetic_time_local_tracking",
        "Repeated recovery of known time-local inverse-operator changes.",
        "tab:synthetic_tracking",
        latex_frame=tracking_summary.rename(columns={
            "time_segment": "seg", "forward_pressure_change": "dF_press",
            "replicates": "n", "A_press_true": "A_press",
            "A_press_est": "Ahat_press", "A_wind_true": "A_wind",
            "A_wind_est": "Ahat_wind", "median_rel_error": "rel_err",
        })[["seg", "dF_press", "n", "A_press", "Ahat_press", "A_wind", "Ahat_wind", "rel_err"]],
    )
    _figure_synthetic(metric_table, task_table, scalar_table, scalar_max_difference, writer)
    import matplotlib.pyplot as plt
    figure, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    axes[0].plot(tracking_summary["time_segment"], tracking_summary["A_press_true"], "o--", label="truth")
    axes[0].plot(tracking_summary["time_segment"], tracking_summary["A_press_est"], "s-", label="ts-AIME")
    axes[0].set(xlabel="Time segment", ylabel="Standardized coefficient", title="Pressure coefficient tracking")
    axes[0].legend(frameon=False)
    axes[1].plot(tracking_summary["time_segment"], tracking_summary["A_wind_true"], "o--", label="truth")
    axes[1].plot(tracking_summary["time_segment"], tracking_summary["A_wind_est"], "s-", label="ts-AIME")
    axes[1].set(xlabel="Time segment", ylabel="Standardized coefficient", title="Wind-U coefficient tracking")
    axes[1].legend(frameon=False)
    for axis in axes: axis.grid(alpha=0.2)
    figure.tight_layout()
    writer.save_figure(figure, "figure01b_synthetic_time_local_tracking")
    return {
        "metrics": metric_table,
        "tasks": task_table,
        "objective": objective_table,
        "scalar_table": scalar_table,
        "tracking": tracking_table,
        "scalar_max_difference": scalar_max_difference,
    }


def _figure_synthetic(
    metrics: pd.DataFrame,
    tasks: pd.DataFrame,
    scalar: pd.DataFrame,
    scalar_difference: float,
    writer: ArtifactWriter,
) -> None:
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    x_axis = metrics["regime"].to_numpy()
    axes[0].scatter(x_axis, metrics["forward_relative_error_SMap"], marker="o", label="S-Map vs true F")
    axes[0].scatter(x_axis, metrics["inverse_relative_error_vector_AIME"], marker="s", label="ts-AIME vs true A")
    axes[0].scatter(x_axis, metrics["inverse_relative_error_marginal_correlations"], marker="^", label="marginal r vs true A")
    axes[0].set(xlabel="Independent replicate", ylabel="Relative Frobenius error", title="Operator recovery across draws")
    axes[0].legend(frameon=False, fontsize=8)
    task_mean = tasks.groupby(["task", "method"])["RMSE"].mean()
    order = [
        ("forward_Y_from_X", "SMap_forward", "S-Map\nforward"),
        ("forward_Y_from_X", "AIME_transpose_forward", r"$A^\top$" + "\nwrong task"),
        ("inverse_X_from_Y", "AIME_inverse", "ts-AIME\ninverse"),
        ("inverse_X_from_Y", "SMap_transpose_inverse", r"$F^\top$" + "\nwrong task"),
        ("inverse_X_from_Y", "marginal_correlation_inverse", "marginal r\ninverse"),
    ]
    values = [float(task_mean.loc[(task, method)]) for task, method, _ in order]
    labels = [label for _, _, label in order]
    axes[1].bar(np.arange(len(values)), values, color=["#0072B2"] * 2 + ["#D55E00"] * 3)
    axes[1].set_xticks(np.arange(len(values)), labels=labels)
    axes[1].set(
        title="Cross-task test",
        ylabel="RMSE",
        xlabel="Blue: forward task; orange: inverse task",
    )
    axes[1].tick_params(axis="x", labelsize=8)
    axes[2].scatter(scalar["Pearson_r"], scalar["scalar_AIME"], color="#009E73", s=45)
    low = min(scalar["Pearson_r"].min(), scalar["scalar_AIME"].min())
    high = max(scalar["Pearson_r"].max(), scalar["scalar_AIME"].max())
    axes[2].plot([low, high], [low, high], "--", color="black", linewidth=1)
    axes[2].set(
        xlabel="Pearson correlation",
        ylabel="Scalar AIME",
        title=f"Scalar identity (max diff {scalar_difference:.1e})",
    )
    for axis in axes:
        axis.grid(alpha=0.2)
    figure.tight_layout()
    writer.save_figure(figure, "figure01_synthetic_operator_validation")


def _select_inverse_ridge(
    inputs: np.ndarray,
    outputs: np.ndarray,
    ridge_grid: Sequence[float],
) -> tuple[float, pd.DataFrame]:
    cut = max(20, int(len(inputs) * 0.65))
    cut = min(cut, len(inputs) - 10)
    rows = []
    for ridge in ridge_grid:
        model = fit_inverse_operator(inputs[:cut], outputs[:cut], ridge=float(ridge))
        reconstructed = model.reconstruct(outputs[cut:])
        rows.append(
            {
                "ridge": float(ridge),
                "inverse_RMSE": float(np.sqrt(np.mean((reconstructed - inputs[cut:]) ** 2))),
            }
        )
    table = pd.DataFrame(rows)
    best = table.sort_values(["inverse_RMSE", "ridge"]).iloc[0]
    return float(best["ridge"]), table


def _baseline_predictions(
    train: pd.DataFrame,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    targets: Sequence[str],
    alpha_grid: Sequence[float],
) -> tuple[dict[str, np.ndarray], pd.DataFrame]:
    """Tune the linear Ridge forecast baseline on validation only."""

    from sklearn.linear_model import Ridge

    train_x_scaler = MatrixStandardizer.fit(train[list(STATE_COLUMNS)].to_numpy(float))
    train_y_scaler = MatrixStandardizer.fit(train[list(targets)].to_numpy(float))
    x_train = train_x_scaler.transform(train[list(STATE_COLUMNS)].to_numpy(float))
    y_train = train_y_scaler.transform(train[list(targets)].to_numpy(float))
    x_validation = train_x_scaler.transform(validation[list(STATE_COLUMNS)].to_numpy(float))
    y_validation = train_y_scaler.transform(validation[list(targets)].to_numpy(float))
    rows = []
    for alpha in alpha_grid:
        model = Ridge(alpha=float(alpha)).fit(x_train, y_train)
        predicted = model.predict(x_validation)
        rows.append(
            {
                "alpha": float(alpha),
                "validation_standardized_RMSE": float(np.sqrt(np.mean((predicted - y_validation) ** 2))),
            }
        )
    selection = pd.DataFrame(rows)
    selected_alpha = float(
        selection.sort_values(["validation_standardized_RMSE", "alpha"]).iloc[0]["alpha"]
    )

    library = pd.concat([train, validation], ignore_index=True)
    x_scaler = MatrixStandardizer.fit(library[list(STATE_COLUMNS)].to_numpy(float))
    y_scaler = MatrixStandardizer.fit(library[list(targets)].to_numpy(float))
    x_library = x_scaler.transform(library[list(STATE_COLUMNS)].to_numpy(float))
    y_library = y_scaler.transform(library[list(targets)].to_numpy(float))
    x_test = x_scaler.transform(test[list(STATE_COLUMNS)].to_numpy(float))
    ridge = Ridge(alpha=selected_alpha).fit(x_library, y_library)
    predictions = {
        "ridge": y_scaler.inverse_transform(ridge.predict(x_test)),
        "persistence": np.repeat(test[["PM25_t"]].to_numpy(float), len(targets), axis=1),
        "training_mean": np.tile(library[list(targets)].mean().to_numpy(float), (len(test), 1)),
    }
    selection["selected"] = selection["alpha"] == selected_alpha
    return predictions, selection


def _fit_quadratic_inverse(
    inputs: np.ndarray,
    outputs: np.ndarray,
    test_outputs: np.ndarray,
    ridge_grid: Sequence[float],
) -> tuple[np.ndarray, pd.DataFrame]:
    """Fit a validation-tuned quadratic inverse diagnostic for linear-gap assessment."""

    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import PolynomialFeatures

    cut = max(20, int(len(inputs) * 0.65))
    cut = min(cut, len(inputs) - 10)
    x_scaler = MatrixStandardizer.fit(inputs[:cut])
    y_scaler = MatrixStandardizer.fit(outputs[:cut])
    x_fit = x_scaler.transform(inputs[:cut])
    x_holdout = x_scaler.transform(inputs[cut:])
    polynomial = PolynomialFeatures(degree=2, include_bias=False)
    y_fit = polynomial.fit_transform(y_scaler.transform(outputs[:cut]))
    y_holdout = polynomial.transform(y_scaler.transform(outputs[cut:]))
    rows = []
    for ridge in ridge_grid:
        model = Ridge(alpha=float(ridge)).fit(y_fit, x_fit)
        predicted = model.predict(y_holdout)
        rows.append(
            {
                "ridge": float(ridge),
                "validation_inverse_RMSE": float(np.sqrt(np.mean((predicted - x_holdout) ** 2))),
            }
        )
    selection = pd.DataFrame(rows)
    selected = float(
        selection.sort_values(["validation_inverse_RMSE", "ridge"]).iloc[0]["ridge"]
    )

    final_x_scaler = MatrixStandardizer.fit(inputs)
    final_y_scaler = MatrixStandardizer.fit(outputs)
    final_polynomial = PolynomialFeatures(degree=2, include_bias=False)
    y_all = final_polynomial.fit_transform(final_y_scaler.transform(outputs))
    model = Ridge(alpha=selected).fit(y_all, final_x_scaler.transform(inputs))
    reconstructed_z = model.predict(
        final_polynomial.transform(final_y_scaler.transform(test_outputs))
    )
    selection["selected"] = selection["ridge"] == selected
    return final_x_scaler.inverse_transform(reconstructed_z), selection


def _inverse_output_subset_diagnostic(
    validation_inputs: np.ndarray,
    validation_outputs: np.ndarray,
    test_inputs: np.ndarray,
    test_outputs: np.ndarray,
    horizons: Sequence[int],
    ridge_grid: Sequence[float],
    *,
    dataset: str,
    site: str,
) -> pd.DataFrame:
    """Test whether the joint forecast vector adds inverse information over one horizon."""

    subsets = [([index], f"{horizon}h_only") for index, horizon in enumerate(horizons)]
    subsets.append((list(range(len(horizons))), "all_horizons"))
    state_scaler = MatrixStandardizer.fit(validation_inputs)
    test_x_z = state_scaler.transform(test_inputs)
    rows = []
    for indices, label in subsets:
        ridge, _ = _select_inverse_ridge(
            validation_inputs,
            validation_outputs[:, indices],
            ridge_grid,
        )
        model = fit_inverse_operator(
            validation_inputs,
            validation_outputs[:, indices],
            ridge=ridge,
        )
        reconstructed_z = state_scaler.transform(model.reconstruct(test_outputs[:, indices]))
        for feature_index, feature in enumerate(STATE_COLUMNS):
            rows.append(
                {
                    "dataset": dataset,
                    "site": site,
                    "output_set": label,
                    "n_outputs": len(indices),
                    "feature": feature,
                    "selected_ridge": ridge,
                    "inverse_RMSE": float(
                        np.sqrt(np.mean((reconstructed_z[:, feature_index] - test_x_z[:, feature_index]) ** 2))
                    ),
                }
            )
    table = pd.DataFrame(rows)
    best_single = (
        table.loc[table["n_outputs"] == 1]
        .groupby("feature")["inverse_RMSE"]
        .min()
    )
    table["gain_all_vs_best_single"] = np.nan
    all_mask = table["output_set"] == "all_horizons"
    table.loc[all_mask, "gain_all_vs_best_single"] = [
        1.0 - row.inverse_RMSE / best_single.loc[row.feature]
        for row in table.loc[all_mask].itertuples()
    ]
    return table


def _meteorology_ablation(
    dataset: str,
    site: str,
    pairs: pd.DataFrame,
    targets: Sequence[str],
    full_predictions: np.ndarray,
    config: APRExperimentConfig,
) -> pd.DataFrame:
    """Compare the full S-Map with pressure/wind ablations under strict OOS."""

    feature_sets = {
        "no_pressure": tuple(value for value in STATE_COLUMNS if value != "PRESS"),
        "no_wind": tuple(value for value in STATE_COLUMNS if value not in {"WIND_U", "WIND_V"}),
        "no_pressure_or_wind": tuple(
            value for value in STATE_COLUMNS if value not in {"PRESS", "WIND_U", "WIND_V"}
        ),
    }
    train = pairs.loc[pairs["split"] == "train"].reset_index(drop=True)
    validation = pairs.loc[pairs["split"] == "validation"].reset_index(drop=True)
    test = pairs.loc[pairs["split"] == "test"].reset_index(drop=True)
    library = pd.concat([train, validation], ignore_index=True)
    observed = test[list(targets)].to_numpy(float)
    block_length = max(2, round(168 / int(config.origin_step_hours)))
    rows: list[dict[str, object]] = []
    for model_name, features in feature_sets.items():
        theta, _ = select_smap_theta(
            train, validation, features, targets, config.theta_grid,
            validation_stride=int(config.validation_stride),
        )
        reduced = run_smap_oos(library, test, features, targets, theta=theta).predictions
        for output_index, horizon in enumerate(config.horizons):
            skill, ci_low, ci_high = moving_block_skill_interval(
                observed[:, output_index],
                full_predictions[:, output_index],
                reduced[:, output_index],
                block_length=block_length,
                n_resamples=int(config.bootstrap_resamples),
                seed=config.seed + 50000 + int(horizon) * 100 + sum(map(ord, site + model_name)) % 997,
            )
            full_metric = regression_metrics(observed[:, output_index], full_predictions[:, output_index])
            reduced_metric = regression_metrics(observed[:, output_index], reduced[:, output_index])
            rows.append(
                {
                    "dataset": dataset,
                    "site": site,
                    "horizon_h": int(horizon),
                    "reduced_model": model_name,
                    "selected_theta_reduced": theta,
                    "full_RMSE": full_metric.rmse,
                    "reduced_RMSE": reduced_metric.rmse,
                    "skill_full_vs_reduced": skill,
                    "skill_CI_low": ci_low,
                    "skill_CI_high": ci_high,
                }
            )
    return pd.DataFrame(rows)


def _pm25_policy_sensitivity(
    frame: pd.DataFrame,
    split: ChronologicalSplit,
    config: APRExperimentConfig,
    *,
    dataset: str,
    site: str,
) -> pd.DataFrame:
    """Re-estimate strict OOS S-Map skill under three PM2.5 negative policies."""

    rows: list[dict[str, object]] = []
    targets = [f"PM25_future_{horizon}h" for horizon in config.horizons]
    for policy in PM25_POLICIES:
        policy_frame = apply_pm25_policy(frame, policy)
        _, pairs, _, _ = prepare_pm25_multihorizon_pairs(
            policy_frame, config.horizons, split,
            origin_step_hours=int(config.origin_step_hours),
            required_states=STATE_COLUMNS,
        )
        train = pairs.loc[pairs["split"] == "train"].reset_index(drop=True)
        validation = pairs.loc[pairs["split"] == "validation"].reset_index(drop=True)
        test = pairs.loc[pairs["split"] == "test"].reset_index(drop=True)
        theta, _ = select_smap_theta(
            train, validation, STATE_COLUMNS, targets, config.theta_grid,
            validation_stride=int(config.validation_stride),
        )
        prediction = run_smap_oos(
            pd.concat([train, validation], ignore_index=True),
            test, STATE_COLUMNS, targets, theta=theta,
        ).predictions
        observed = test[targets].to_numpy(float)
        persistence = np.repeat(test[["PM25_t"]].to_numpy(float), len(targets), axis=1)
        block_length = max(2, round(168 / int(config.origin_step_hours)))
        for output_index, horizon in enumerate(config.horizons):
            skill, ci_low, ci_high = moving_block_skill_interval(
                observed[:, output_index], prediction[:, output_index], persistence[:, output_index],
                block_length=block_length, n_resamples=int(config.bootstrap_resamples),
                seed=config.seed + 70000 + int(horizon) * 100 + PM25_POLICIES.index(policy),
            )
            metric = regression_metrics(observed[:, output_index], prediction[:, output_index])
            rows.append(
                {
                    "dataset": dataset, "site": site, "policy": policy,
                    "horizon_h": int(horizon), "selected_theta": theta,
                    "test_n": metric.n, "RMSE": metric.rmse, "rho": metric.rho,
                    "skill_vs_persistence": skill,
                    "skill_CI_low": ci_low, "skill_CI_high": ci_high,
                }
            )
    return pd.DataFrame(rows)


def _rolling_inverse_oos(
    dates: pd.Series,
    input_z: np.ndarray,
    output_z: np.ndarray,
    *,
    ridge: float,
    dataset: str,
    site: str,
    window_days: int,
    step_days: int,
    min_samples: int,
    origin_step_hours: int,
    bootstrap_resamples: int,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fit on a trailing window and evaluate reconstruction on the next block."""

    date_index = pd.DatetimeIndex(pd.to_datetime(dates))
    endpoint = date_index.min() + pd.Timedelta(days=window_days)
    observations: dict[int, list[np.ndarray]] = {i: [] for i in range(input_z.shape[1])}
    aime_predictions: dict[int, list[np.ndarray]] = {i: [] for i in range(input_z.shape[1])}
    marginal_predictions: dict[int, list[np.ndarray]] = {i: [] for i in range(input_z.shape[1])}
    window_rows: list[dict[str, object]] = []
    while endpoint + pd.Timedelta(days=step_days) <= date_index.max():
        calibration_start = endpoint - pd.Timedelta(days=window_days)
        evaluation_end = endpoint + pd.Timedelta(days=step_days)
        calibration = np.flatnonzero((date_index > calibration_start) & (date_index <= endpoint))
        evaluation = np.flatnonzero((date_index > endpoint) & (date_index <= evaluation_end))
        if len(calibration) >= min_samples and len(evaluation) >= 3:
            model = fit_inverse_operator(
                input_z[calibration], output_z[calibration], ridge=ridge
            )
            aime = model.reconstruct(output_z[evaluation])
            output_local = model.y_scaler.transform(output_z[evaluation])
            marginal_local = output_local @ model.cross_covariance.T
            marginal = model.x_scaler.inverse_transform(marginal_local)
            for feature_index, feature in enumerate(STATE_COLUMNS):
                observed = input_z[evaluation, feature_index]
                observations[feature_index].append(observed)
                aime_predictions[feature_index].append(aime[:, feature_index])
                marginal_predictions[feature_index].append(marginal[:, feature_index])
                window_rows.append(
                    {
                        "dataset": dataset,
                        "site": site,
                        "calibration_start": calibration_start,
                        "calibration_end": endpoint,
                        "evaluation_start": endpoint,
                        "evaluation_end": evaluation_end,
                        "calibration_n": len(calibration),
                        "evaluation_n": len(evaluation),
                        "feature": feature,
                        "AIME_RMSE": float(np.sqrt(np.mean((aime[:, feature_index] - observed) ** 2))),
                        "marginal_RMSE": float(np.sqrt(np.mean((marginal[:, feature_index] - observed) ** 2))),
                    }
                )
        endpoint += pd.Timedelta(days=step_days)

    summary_rows = []
    block_length = max(2, round(168 / origin_step_hours))
    for feature_index, feature in enumerate(STATE_COLUMNS):
        if not observations[feature_index]:
            continue
        observed = np.concatenate(observations[feature_index])
        aime = np.concatenate(aime_predictions[feature_index])
        marginal = np.concatenate(marginal_predictions[feature_index])
        skill, low, high = moving_block_skill_interval(
            observed,
            aime,
            marginal,
            block_length=block_length,
            n_resamples=bootstrap_resamples,
            seed=seed + feature_index,
        )
        summary_rows.append(
            {
                "dataset": dataset,
                "site": site,
                "feature": feature,
                "evaluation_n": len(observed),
                "AIME_RMSE": regression_metrics(observed, aime).rmse,
                "marginal_RMSE": regression_metrics(observed, marginal).rmse,
                "skill_AIME_vs_marginal": skill,
                "skill_CI_low": low,
                "skill_CI_high": high,
            }
        )
    return pd.DataFrame(summary_rows), pd.DataFrame(window_rows)


def _rolling_bridge(
    dates: pd.Series,
    input_z: np.ndarray,
    output_z: np.ndarray,
    coefficients: np.ndarray,
    *,
    ridge: float,
    dataset: str,
    site: str,
    horizons: Sequence[int],
    window_days: int,
    step_days: int,
    min_samples: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    date_index = pd.DatetimeIndex(pd.to_datetime(dates))
    endpoint = date_index.min() + pd.Timedelta(days=window_days)
    rows = []
    operators = []
    while endpoint <= date_index.max():
        start = endpoint - pd.Timedelta(days=window_days)
        positions = np.flatnonzero((date_index > start) & (date_index <= endpoint))
        if len(positions) >= min_samples:
            x_window = input_z[positions]
            y_window = output_z[positions]
            empirical = fit_inverse_operator(x_window, y_window, ridge=ridge).operator
            x_local = MatrixStandardizer.fit(x_window).transform(x_window)
            y_local = MatrixStandardizer.fit(y_window).transform(y_window)
            covariance = np.cov(x_local, rowvar=False, ddof=0)
            local_coefficients = coefficients_in_window_coordinates(
                coefficients[positions], x_window, y_window
            )
            forward = np.nanmedian(local_coefficients, axis=0)
            residual = y_local - x_local @ forward.T
            residual_covariance = np.cov(residual, rowvar=False, ddof=0)
            implied = covariance_weighted_inverse(
                forward, covariance, ridge=ridge,
                output_noise_covariance=residual_covariance,
            )
            unregularized = fit_inverse_operator(x_window, y_window, ridge=0.0).operator
            closure = float(
                np.linalg.norm(forward @ unregularized - np.eye(len(horizons)))
                / np.linalg.norm(np.eye(len(horizons)))
            )
            projection = unregularized @ forward
            idempotence = float(
                np.linalg.norm(projection @ projection - projection)
                / max(np.linalg.norm(projection), 1e-12)
            )
            rows.append(
                {
                    "dataset": dataset,
                    "site": site,
                    "endpoint": endpoint,
                    "n_window": len(positions),
                    "window_days": window_days,
                    "ridge": ridge,
                    "AIME_vs_SMap_implied_cosine": operator_cosine(empirical, implied),
                    "AIME_vs_SMap_implied_relative_error": float(
                        np.linalg.norm(empirical - implied) / max(np.linalg.norm(implied), 1e-12)
                    ),
                    "forward_inverse_closure_error_unregularized": closure,
                    "input_projection_idempotence_error_unregularized": idempotence,
                    "operator_frobenius_norm": float(np.linalg.norm(empirical)),
                    "estimated_output_noise_trace": float(np.trace(residual_covariance)),
                }
            )
            for feature_index, feature in enumerate(STATE_COLUMNS):
                for output_index, horizon in enumerate(horizons):
                    operators.append(
                        {
                            "dataset": dataset,
                            "site": site,
                            "endpoint": endpoint,
                            "season": _regime_season(endpoint),
                            "window_days": window_days,
                            "feature": feature,
                            "horizon_h": int(horizon),
                            "AIME": float(empirical[feature_index, output_index]),
                            "SMap_implied_inverse": float(implied[feature_index, output_index]),
                        }
                    )
        endpoint += pd.Timedelta(days=step_days)
    return pd.DataFrame(rows), pd.DataFrame(operators)


def _run_real_site(
    dataset: str,
    site: str,
    pairs: pd.DataFrame,
    targets: Sequence[str],
    config: APRExperimentConfig,
) -> dict[str, Any]:
    train = pairs.loc[pairs["split"] == "train"].reset_index(drop=True)
    validation = pairs.loc[pairs["split"] == "validation"].reset_index(drop=True)
    test = pairs.loc[pairs["split"] == "test"].reset_index(drop=True)
    theta, theta_table = select_smap_theta(
        train,
        validation,
        STATE_COLUMNS,
        targets,
        config.theta_grid,
        validation_stride=int(config.validation_stride),
    )
    library = pd.concat([train, validation], ignore_index=True)
    smap_test = run_smap_oos(
        library,
        test,
        STATE_COLUMNS,
        targets,
        theta=theta,
    )
    observed = test[list(targets)].to_numpy(float)
    baselines, forecast_ridge_selection = _baseline_predictions(
        train, validation, test, targets, config.forecast_ridge_grid
    )
    forecast_rows = []
    block_length = max(2, round(168 / int(config.origin_step_hours)))
    models = {"SMap": smap_test.predictions, **baselines}
    for output_index, horizon in enumerate(config.horizons):
        persistence = baselines["persistence"][:, output_index]
        for model_name, predictions in models.items():
            point = regression_metrics(observed[:, output_index], predictions[:, output_index])
            skill, ci_low, ci_high = moving_block_skill_interval(
                observed[:, output_index],
                predictions[:, output_index],
                persistence,
                block_length=block_length,
                n_resamples=int(config.bootstrap_resamples),
                seed=config.seed + int(horizon) * 100 + sum(map(ord, site)) % 997,
            )
            forecast_rows.append(
                {
                    "dataset": dataset,
                    "site": site,
                    "horizon_h": int(horizon),
                    "model": model_name,
                    **point.as_dict(),
                    "skill_vs_persistence": skill,
                    "skill_CI_low": ci_low,
                    "skill_CI_high": ci_high,
                }
            )

    forecast_comparison_rows = []
    for output_index, horizon in enumerate(config.horizons):
        skill, ci_low, ci_high = moving_block_skill_interval(
            observed[:, output_index],
            smap_test.predictions[:, output_index],
            baselines["ridge"][:, output_index],
            block_length=block_length,
            n_resamples=int(config.bootstrap_resamples),
            seed=config.seed + 30000 + int(horizon) * 100 + sum(map(ord, site)) % 997,
        )
        forecast_comparison_rows.append({
            "dataset": dataset, "site": site, "horizon_h": int(horizon),
            "selected_ridge_alpha": float(
                forecast_ridge_selection.loc[forecast_ridge_selection["selected"], "alpha"].iloc[0]
            ),
            "skill_SMap_vs_tuned_ridge": skill,
            "skill_CI_low": ci_low, "skill_CI_high": ci_high,
        })
    forecast_ridge_selection.insert(0, "dataset", dataset)
    forecast_ridge_selection.insert(1, "site", site)

    smap_validation = run_smap_oos(
        train,
        validation,
        STATE_COLUMNS,
        targets,
        theta=theta,
    )
    inverse_ridge, ridge_table = _select_inverse_ridge(
        validation[list(STATE_COLUMNS)].to_numpy(float),
        smap_validation.predictions,
        config.inverse_ridge_grid,
    )
    inverse_model = fit_inverse_operator(
        validation[list(STATE_COLUMNS)].to_numpy(float),
        smap_validation.predictions,
        ridge=inverse_ridge,
    )
    reconstructed = inverse_model.reconstruct(smap_test.predictions)
    final_state_scaler = MatrixStandardizer.fit(library[list(STATE_COLUMNS)].to_numpy(float))
    reconstructed_z = final_state_scaler.transform(reconstructed)
    test_x_z = final_state_scaler.transform(test[list(STATE_COLUMNS)].to_numpy(float))

    quadratic_reconstructed, quadratic_selection = _fit_quadratic_inverse(
        validation[list(STATE_COLUMNS)].to_numpy(float),
        smap_validation.predictions,
        smap_test.predictions,
        config.quadratic_ridge_grid,
    )
    quadratic_reconstructed_z = final_state_scaler.transform(quadratic_reconstructed)
    quadratic_selection.insert(0, "dataset", dataset)
    quadratic_selection.insert(1, "site", site)
    multihorizon = _inverse_output_subset_diagnostic(
        validation[list(STATE_COLUMNS)].to_numpy(float),
        smap_validation.predictions,
        test[list(STATE_COLUMNS)].to_numpy(float),
        smap_test.predictions,
        config.horizons,
        config.inverse_ridge_grid,
        dataset=dataset, site=site,
    )

    marginal = inverse_model.cross_covariance
    test_output_z_validation = inverse_model.y_scaler.transform(smap_test.predictions)
    marginal_x_z_validation = test_output_z_validation @ marginal.T
    marginal_raw = inverse_model.x_scaler.inverse_transform(marginal_x_z_validation)
    marginal_z = final_state_scaler.transform(marginal_raw)
    inverse_rows = []
    for feature_index, feature in enumerate(STATE_COLUMNS):
        linear_skill, linear_ci_low, linear_ci_high = moving_block_skill_interval(
            test_x_z[:, feature_index],
            reconstructed_z[:, feature_index],
            quadratic_reconstructed_z[:, feature_index],
            block_length=block_length,
            n_resamples=int(config.bootstrap_resamples),
            seed=config.seed + 40000 + feature_index + sum(map(ord, site)) % 997,
        )
        inverse_rows.append(
            {
                "dataset": dataset,
                "site": site,
                "feature": feature,
                "ridge": inverse_ridge,
                "AIME_inverse_RMSE": float(
                    np.sqrt(np.mean((reconstructed_z[:, feature_index] - test_x_z[:, feature_index]) ** 2))
                ),
                "marginal_correlation_inverse_RMSE": float(
                    np.sqrt(np.mean((marginal_z[:, feature_index] - test_x_z[:, feature_index]) ** 2))
                ),
                "AIME_inverse_rho": regression_metrics(
                    test_x_z[:, feature_index], reconstructed_z[:, feature_index]
                ).rho,
                "quadratic_inverse_RMSE": float(np.sqrt(np.mean((
                    quadratic_reconstructed_z[:, feature_index] - test_x_z[:, feature_index]
                ) ** 2))),
                "linear_skill_vs_quadratic": linear_skill,
                "linear_vs_quadratic_CI_low": linear_ci_low,
                "linear_vs_quadratic_CI_high": linear_ci_high,
            }
        )
    rolling_inverse, rolling_inverse_windows = _rolling_inverse_oos(
        test["Date"], test_x_z, smap_test.predictions_standardized,
        ridge=inverse_ridge, dataset=dataset, site=site,
        window_days=config.aime_window_days,
        step_days=int(config.aime_endpoint_step_days),
        min_samples=max(30 if config.run_mode == "publication" else 15, 5 * len(config.horizons)),
        origin_step_hours=int(config.origin_step_hours),
        bootstrap_resamples=int(config.bootstrap_resamples),
        seed=config.seed + 45000 + sum(map(ord, dataset + site)),
    )
    diagnostics, operators = _rolling_bridge(
        test["Date"],
        test_x_z,
        smap_test.predictions_standardized,
        smap_test.coefficients_standardized,
        ridge=inverse_ridge,
        dataset=dataset,
        site=site,
        horizons=config.horizons,
        window_days=config.aime_window_days,
        step_days=int(config.aime_endpoint_step_days),
        min_samples=max(30 if config.run_mode == "publication" else 15, 5 * len(config.horizons)),
    )
    observed_norm, null_p, null_values = structured_period_shift_test(
        test["Date"],
        test_x_z,
        smap_test.predictions_standardized,
        ridge=inverse_ridge,
        sampling_interval_hours=int(config.origin_step_hours),
        period_hours=168,
        n_permutations=config.null_permutations,
        seed=config.seed + sum(map(ord, dataset + site)),
    )
    null_table = pd.DataFrame(
        [
            {
                "dataset": dataset,
                "site": site,
                "observed_operator_norm": observed_norm,
                "exact_weekly_phase_shift_p_value": null_p,
                "null_unique_shifts": len(null_values),
                "minimum_attainable_p_value": 1.0 / (1.0 + len(null_values)),
                "null_shift_mode": "exhaustive" if config.null_permutations is None else "without_replacement_subset",
                "null_median": float(np.nanmedian(null_values)),
                "null_95pct": float(np.nanpercentile(null_values, 95)),
            }
        ]
    )
    theta_table.insert(0, "dataset", dataset)
    theta_table.insert(1, "site", site)
    ridge_table.insert(0, "dataset", dataset)
    ridge_table.insert(1, "site", site)
    return {
        "forecast": pd.DataFrame(forecast_rows),
        "forecast_comparison": pd.DataFrame(forecast_comparison_rows),
        "forecast_ridge_selection": forecast_ridge_selection,
        "inverse": pd.DataFrame(inverse_rows),
        "quadratic_selection": quadratic_selection,
        "multihorizon_inverse": multihorizon,
        "rolling_inverse_oos": rolling_inverse,
        "rolling_inverse_windows": rolling_inverse_windows,
        "diagnostics": diagnostics,
        "operators": operators,
        "null": null_table,
        "theta": theta_table,
        "ridge_selection": ridge_table,
        "test_dates": test["Date"].reset_index(drop=True),
        "test_input_z": test_x_z,
        "test_prediction_z": smap_test.predictions_standardized,
        "test_predictions": smap_test.predictions,
        "test_observed": observed,
    }


def _concatenate(results: dict[tuple[str, str], dict[str, Any]], key: str) -> pd.DataFrame:
    frames = [result[key] for result in results.values() if key in result and len(result[key])]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _figure_data_coverage(audit: pd.DataFrame, writer: ArtifactWriter) -> None:
    if audit.empty:
        return
    import matplotlib.pyplot as plt

    coverage = audit.pivot_table(
        index=["dataset", "site"], columns="year", values="all_required_coverage"
    )
    figure, axis = plt.subplots(figsize=(11, max(4.5, 0.38 * len(coverage) + 2)))
    image = axis.imshow(coverage.to_numpy(), aspect="auto", vmin=0, vmax=1, cmap="viridis")
    axis.set_xticks(np.arange(len(coverage.columns)), labels=coverage.columns.astype(str))
    axis.set_yticks(
        np.arange(len(coverage.index)), labels=[f"{dataset}: {site}" for dataset, site in coverage.index]
    )
    axis.set_xlabel("Year")
    axis.set_title("Complete hourly coverage of PM2.5 and common meteorological variables")
    figure.colorbar(image, ax=axis, pad=0.01, label="Complete-case proportion")
    figure.tight_layout()
    writer.save_figure(figure, "figure02_data_coverage")


## 6. Publication figures, tables, readiness checks, and ZIP output


In [ ]:
def _compact_forecast_table(forecast: pd.DataFrame) -> pd.DataFrame:
    shown = forecast.loc[forecast["model"] == "SMap"].copy()
    return shown.groupby(["dataset", "horizon_h"], as_index=False).agg(
        sites=("site", "nunique"),
        median_RMSE=("RMSE", "median"),
        median_skill=("skill_vs_persistence", "median"),
        q25_skill=("skill_vs_persistence", lambda values: values.quantile(0.25)),
        q75_skill=("skill_vs_persistence", lambda values: values.quantile(0.75)),
        positive_CI_sites=("skill_CI_low", lambda values: int((values > 0).sum())),
    )


def _compact_inverse_table(inverse: pd.DataFrame) -> pd.DataFrame:
    return inverse.groupby(["dataset", "feature"], as_index=False).agg(
        sites=("site", "nunique"),
        median_AIME_RMSE=("AIME_inverse_RMSE", "median"),
        median_marginal_RMSE=("marginal_correlation_inverse_RMSE", "median"),
        AIME_wins=("AIME_inverse_RMSE", lambda values: int((values.to_numpy() < inverse.loc[values.index, "marginal_correlation_inverse_RMSE"].to_numpy()).sum())),
    )


def _compact_bridge_table(diagnostics: pd.DataFrame) -> pd.DataFrame:
    return diagnostics.groupby("dataset", as_index=False).agg(
        sites=("site", "nunique"),
        median_inverse_cosine=("AIME_vs_SMap_implied_cosine", "median"),
        median_relative_error=("AIME_vs_SMap_implied_relative_error", "median"),
        median_closure_error=("forward_inverse_closure_error_unregularized", "median"),
    )


def _compact_theta_table(theta: pd.DataFrame, maximum_theta: float) -> pd.DataFrame:
    selected = theta.loc[theta.groupby(["dataset", "site"])["validation_standardized_RMSE"].idxmin()].copy()
    selected["at_upper_boundary"] = selected["theta"] == maximum_theta
    return selected.groupby("dataset", as_index=False).agg(
        sites=("site", "nunique"),
        median_selected_theta=("theta", "median"),
        min_selected_theta=("theta", "min"),
        max_selected_theta=("theta", "max"),
        sites_at_upper_boundary=("at_upper_boundary", "sum"),
    )


def _compact_ridge_table(ridge: pd.DataFrame) -> pd.DataFrame:
    selected = ridge.loc[ridge.groupby(["dataset", "site"])["inverse_RMSE"].idxmin()].copy()
    return selected.groupby("dataset", as_index=False).agg(
        sites=("site", "nunique"),
        median_selected_ridge=("ridge", "median"),
        median_validation_inverse_RMSE=("inverse_RMSE", "median"),
    )


def _publication_figures(
    forecast: pd.DataFrame,
    diagnostics: pd.DataFrame,
    operators: pd.DataFrame,
    results: dict[tuple[str, str], dict[str, Any]],
    ablation: pd.DataFrame,
    config: APRExperimentConfig,
    writer: ArtifactWriter,
) -> None:
    import matplotlib.pyplot as plt

    if not forecast.empty:
        shown = forecast.loc[forecast["model"] == "SMap"].copy()
        shown["site_label"] = shown["dataset"] + ": " + shown["site"]
        sites = list(dict.fromkeys(shown["site_label"]))
        figure, axes = plt.subplots(
            1,
            len(config.horizons),
            figsize=(15, max(5, 0.28 * len(sites) + 2)),
            sharey=True,
        )
        axes = np.atleast_1d(axes)
        for axis, horizon in zip(axes, config.horizons):
            part = shown.loc[shown["horizon_h"] == horizon].set_index("site_label").reindex(sites)
            y_position = np.arange(len(sites))
            axis.errorbar(
                part["skill_vs_persistence"],
                y_position,
                xerr=np.vstack(
                    [
                        part["skill_vs_persistence"] - part["skill_CI_low"],
                        part["skill_CI_high"] - part["skill_vs_persistence"],
                    ]
                ),
                fmt="o",
                color="#0072B2",
                ecolor="#7AA6C2",
                capsize=2,
            )
            axis.axvline(0, color="black", linewidth=0.8)
            axis.set_title(f"{horizon}-hour forecast")
            axis.set_xlabel("RMSE skill vs persistence")
            axis.grid(axis="x", alpha=0.2)
            axis.set_yticks(y_position, labels=sites)
        figure.suptitle("Strict out-of-sample S-Map skill with moving-block 95% confidence intervals")
        figure.tight_layout()
        writer.save_figure(figure, "figure03_forecast_skill_forest")

    if not diagnostics.empty:
        summary = diagnostics.groupby(["dataset", "site"], as_index=False).agg(
            median_inverse_cosine=("AIME_vs_SMap_implied_cosine", "median"),
            q25_inverse_cosine=("AIME_vs_SMap_implied_cosine", lambda values: values.quantile(0.25)),
            q75_inverse_cosine=("AIME_vs_SMap_implied_cosine", lambda values: values.quantile(0.75)),
            median_bridge_relative_error=("AIME_vs_SMap_implied_relative_error", "median"),
            median_closure_error=("forward_inverse_closure_error_unregularized", "median"),
            q25_closure_error=("forward_inverse_closure_error_unregularized", lambda values: values.quantile(0.25)),
            q75_closure_error=("forward_inverse_closure_error_unregularized", lambda values: values.quantile(0.75)),
        )
        writer.save_table(
            summary,
            "table14_operator_bridge_by_site",
            "Median operator bridge diagnostics by site.",
            "tab:operator_bridge_site",
            latex_frame=summary.rename(columns={
                "median_inverse_cosine": "cosine",
                "median_bridge_relative_error": "bridge_error",
                "median_closure_error": "closure_error",
            })[["dataset", "site", "cosine", "bridge_error", "closure_error"]],
        )
        labels = (summary["dataset"] + ": " + summary["site"]).tolist()
        y_position = np.arange(len(labels))
        figure, axes = plt.subplots(1, 2, figsize=(12, max(4.5, 0.3 * len(summary) + 1.5)))
        axes[0].barh(y_position, summary["median_inverse_cosine"], color="#009E73")
        axes[0].errorbar(summary["median_inverse_cosine"], y_position, xerr=np.vstack([summary["median_inverse_cosine"] - summary["q25_inverse_cosine"], summary["q75_inverse_cosine"] - summary["median_inverse_cosine"]]), fmt="none", color="black", capsize=2)
        axes[0].set_yticks(y_position, labels=labels)
        axes[0].set(xlabel="Cosine similarity", title="Empirical AIME vs S-Map-implied inverse")
        axes[0].axvline(0, color="black", linewidth=0.8)
        axes[1].barh(y_position, summary["median_closure_error"], color="#D55E00")
        axes[1].errorbar(summary["median_closure_error"], y_position, xerr=np.vstack([summary["median_closure_error"] - summary["q25_closure_error"], summary["q75_closure_error"] - summary["median_closure_error"]]), fmt="none", color="black", capsize=2)
        axes[1].set_yticks(y_position, labels=[])
        axes[1].set(
            xlabel="Relative error",
            title=r"Forward-inverse closure $\|FA-I\|_F/\|I\|_F$",
        )
        for axis in axes:
            axis.grid(axis="x", alpha=0.2)
        figure.tight_layout()
        writer.save_figure(figure, "figure04_forward_inverse_bridge")

    if operators.empty:
        return
    seasonal = operators.groupby(
        ["dataset", "site", "season", "feature", "horizon_h"], as_index=False
    ).agg(
        mean_AIME=("AIME", "mean"),
        median_AIME=("AIME", "median"),
        mean_abs_AIME=("AIME", lambda values: float(np.mean(np.abs(values)))),
        n_windows=("AIME", "size"),
    )
    atmospheric_features = ("RH", "PRESS", "RAIN_EVENT", "WIND_U", "WIND_V")
    seasonal_compact = seasonal.loc[seasonal["feature"].isin(atmospheric_features)].groupby(
        ["dataset", "feature", "horizon_h"], as_index=False
    ).agg(median_AIME=("median_AIME", "median"), n_site_seasons=("site", "size"))
    writer.save_table(
        seasonal,
        "table15_seasonal_vector_aime",
        "Seasonal vector-output ts-AIME summaries.",
        "tab:seasonal_aime",
        latex_frame=seasonal_compact,
    )
    matrices = []
    for dataset in operators["dataset"].drop_duplicates():
        part = operators.loc[operators["dataset"] == dataset]
        matrix = (
            part.groupby(["feature", "horizon_h"])["AIME"]
            .median()
            .unstack("horizon_h")
            .reindex(STATE_COLUMNS)
        )
        matrices.append((dataset, matrix))
    if matrices:
        limit = max(float(np.nanmax(np.abs(matrix.to_numpy()))) for _, matrix in matrices) or 1.0
        figure, axes = plt.subplots(
            len(matrices),
            1,
            figsize=(10, 3.8 * len(matrices)),
            squeeze=False,
            constrained_layout=True,
        )
        for axis, (dataset, matrix) in zip(axes[:, 0], matrices):
            image = axis.imshow(
                matrix.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-limit, vmax=limit
            )
            axis.set_xticks(
                np.arange(len(config.horizons)), labels=[f"{horizon} h" for horizon in config.horizons]
            )
            axis.set_yticks(np.arange(len(STATE_COLUMNS)), labels=STATE_COLUMNS)
            n_sites = operators.loc[operators["dataset"] == dataset, "site"].nunique()
            axis.set_title(f"{dataset}: median across {n_sites} site(s) and rolling windows")
        figure.colorbar(
            image,
            ax=axes[:, 0].tolist(),
            pad=0.02,
            label="Median standardized inverse coefficient (common scale)",
        )
        figure.suptitle("Cross-region multi-horizon ts-AIME operators")
        writer.save_figure(figure, "figure05_cross_region_vector_aime")

        atmospheric = operators.loc[operators["feature"].isin(atmospheric_features)].groupby(
            ["dataset", "feature", "horizon_h"], as_index=False
        ).agg(
            median_AIME=("AIME", "median"),
            q25_AIME=("AIME", lambda values: values.quantile(0.25)),
            q75_AIME=("AIME", lambda values: values.quantile(0.75)),
        )
        figure, axes = plt.subplots(
            len(matrices),
            1,
            figsize=(10, 3.8 * len(matrices)),
            squeeze=False,
        )
        for axis, (dataset, _) in zip(axes[:, 0], matrices):
            part = atmospheric.loc[atmospheric["dataset"] == dataset]
            for feature in atmospheric_features:
                values = part.loc[part["feature"] == feature].sort_values("horizon_h")
                if values.empty:
                    continue
                axis.errorbar(
                    values["horizon_h"], values["median_AIME"],
                    yerr=np.vstack([values["median_AIME"] - values["q25_AIME"], values["q75_AIME"] - values["median_AIME"]]),
                    marker="o", capsize=2, label=feature,
                )
            axis.axhline(0, color="black", linewidth=0.8)
            axis.set_ylabel("Median AIME (IQR)")
            axis.set_title(f"{dataset}: horizon-specific atmospheric alignment")
            axis.grid(alpha=0.2)
            axis.legend(frameon=False, ncol=5, fontsize=8)
        axes[-1, 0].set_xlabel("Forecast horizon (hours)")
        figure.tight_layout()
        writer.save_figure(figure, "figure06_rolling_atmospheric_alignment")

    if not ablation.empty:
        datasets = ablation["dataset"].drop_duplicates().tolist()
        figure, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 4.2), squeeze=False)
        for axis, dataset in zip(axes[0], datasets):
            part = ablation.loc[ablation["dataset"] == dataset]
            for model_name in part["reduced_model"].drop_duplicates():
                values = part.loc[part["reduced_model"] == model_name].groupby("horizon_h", as_index=False).agg(
                    median_skill=("skill_full_vs_reduced", "median"),
                    q25=("skill_full_vs_reduced", lambda series: series.quantile(0.25)),
                    q75=("skill_full_vs_reduced", lambda series: series.quantile(0.75)),
                )
                axis.plot(values["horizon_h"], values["median_skill"], marker="o", label=model_name)
                axis.fill_between(values["horizon_h"], values["q25"], values["q75"], alpha=0.12)
            axis.axhline(0, color="black", linewidth=0.8)
            axis.set(xlabel="Forecast horizon (hours)", ylabel="RMSE skill: full vs reduced", title=dataset)
            axis.grid(alpha=0.2)
            axis.legend(frameon=False, fontsize=8)
        figure.suptitle("Strict OOS pressure and wind ablation")
        figure.tight_layout()
        writer.save_figure(figure, "figure07_meteorology_ablation")


def _window_sensitivity(
    results: dict[tuple[str, str], dict[str, Any]],
    config: APRExperimentConfig,
) -> pd.DataFrame:
    rows = []
    output_names = [f"prediction_{h}h" for h in config.horizons]
    for (dataset, site), result in results.items():
        aligned = pd.DataFrame(result["test_input_z"], columns=STATE_COLUMNS)
        for index, output in enumerate(output_names):
            aligned[output] = result["test_prediction_z"][:, index]
        aligned["Date"] = result["test_dates"].to_numpy()
        ridge = float(result["inverse"]["ridge"].iloc[0])
        for window_days in config.aime_window_sensitivity_days:
            estimator = RollingVectorTSAIME(
                RollingVectorTSAIMEConfig(
                    window=f"{int(window_days)}D",
                    step=f"{int(config.aime_endpoint_step_days)}D",
                    ridge=ridge,
                    min_samples=max(
                        30 if config.run_mode == "publication" else 15,
                        5 * len(config.horizons),
                    ),
                )
            )
            fitted = estimator.fit(aligned, STATE_COLUMNS, output_names)
            if not fitted.diagnostics.empty:
                rows.append(
                    {
                        "dataset": dataset,
                        "site": site,
                        "window_days": int(window_days),
                        "median_operator_norm": float(
                            fitted.diagnostics["operator_frobenius_norm"].median()
                        ),
                        "n_windows": int(len(fitted.diagnostics)),
                    }
                )
    return pd.DataFrame(rows)


def _readiness_report(
    config: APRExperimentConfig,
    run_id: str,
    synthetic: dict[str, Any],
    results: dict[tuple[str, str], dict[str, Any]],
    forecast: pd.DataFrame,
    theta: pd.DataFrame,
    nulls: pd.DataFrame,
    ablation: pd.DataFrame,
    pm25_policy: pd.DataFrame,
    writer: ArtifactWriter,
) -> pd.DataFrame:
    problems: list[str] = []
    notes: list[str] = []
    notes.append("NIES license notice discrepancy requires manual verification before public release")
    scalar_difference = float(synthetic["scalar_max_difference"])
    if scalar_difference > 1e-10:
        problems.append(f"scalar equivalence error {scalar_difference:.3e} exceeds 1e-10")
    metrics = synthetic["metrics"]
    if metrics["forward_relative_error_SMap"].median() > 0.20:
        problems.append("median S-Map forward-operator recovery error exceeds 0.20")
    if metrics["inverse_relative_error_vector_AIME"].median() > 0.20:
        problems.append("median vector-AIME inverse-operator recovery error exceeds 0.20")
    if not (
        metrics["inverse_relative_error_vector_AIME"].median()
        < metrics["inverse_relative_error_marginal_correlations"].median()
    ):
        problems.append("vector AIME did not improve inverse recovery over marginal correlations")
    tracking = synthetic["tracking"]
    if tracking["inverse_relative_error"].max() > 0.20:
        problems.append("time-local inverse tracking error exceeds 0.20")
    completed = pd.DataFrame([{"dataset": dataset, "site": site} for dataset, site in results])
    beijing_n = int((completed["dataset"] == "Beijing").sum()) if len(completed) else 0
    tsukuba_n = int((completed["dataset"] == "Tsukuba").sum()) if len(completed) else 0
    thai_n = int((completed["dataset"] == "Thailand").sum()) if len(completed) else 0
    required_beijing = 10 if config.run_mode == "publication" else 1
    if beijing_n < required_beijing:
        problems.append(f"only {beijing_n} Beijing sites completed; required {required_beijing}")
    if tsukuba_n < 1:
        problems.append("required Tsukuba experiment did not complete")
    selected_theta = theta.loc[theta.groupby(["dataset", "site"])["validation_standardized_RMSE"].idxmin()] if not theta.empty else pd.DataFrame()
    upper_boundary_n = int((selected_theta["theta"] == max(config.theta_grid)).sum()) if not selected_theta.empty else 0
    if upper_boundary_n and config.run_mode == "publication":
        problems.append(f"{upper_boundary_n} site(s) selected the upper theta boundary {max(config.theta_grid)}")
    elif upper_boundary_n:
        notes.append(f"smoke run selected the upper theta boundary at {upper_boundary_n} site(s)")
    forecast_ridge = _concatenate(results, "forecast_ridge_selection")
    forecast_ridge_upper_n = int((
        forecast_ridge.loc[forecast_ridge["selected"], "alpha"] == max(config.forecast_ridge_grid)
    ).sum()) if not forecast_ridge.empty else 0
    quadratic_ridge = _concatenate(results, "quadratic_selection")
    quadratic_ridge_upper_n = int((
        quadratic_ridge.loc[quadratic_ridge["selected"], "ridge"] == max(config.quadratic_ridge_grid)
    ).sum()) if not quadratic_ridge.empty else 0
    if forecast_ridge_upper_n:
        problems.append(f"{forecast_ridge_upper_n} forecast Ridge fit(s) selected the upper alpha boundary")
    if quadratic_ridge_upper_n:
        problems.append(f"{quadratic_ridge_upper_n} quadratic inverse fit(s) selected the upper ridge boundary")
    critical_skill = forecast.loc[(forecast["model"] == "SMap") & (forecast["horizon_h"].isin([6, 24]))] if not forecast.empty else pd.DataFrame()
    if config.run_mode == "publication" and (critical_skill.empty or (critical_skill["skill_CI_low"] <= 0).any()):
        problems.append("not every 6 h and 24 h S-Map skill interval is positive")
    expected_ablation_rows = len(results) * 3 * len(config.horizons)
    if len(ablation) != expected_ablation_rows:
        problems.append(f"meteorology ablation has {len(ablation)} rows; expected {expected_ablation_rows}")
    if tsukuba_n and len(pm25_policy) != len(PM25_POLICIES) * len(config.horizons):
        problems.append("Tsukuba PM2.5 policy sensitivity is incomplete")
    if not nulls.empty and (nulls["null_unique_shifts"] < 1).any():
        problems.append("structured null did not produce unique admissible shifts")
    if config.run_mode == "publication" and config.null_permutations is not None:
        problems.append("publication mode requires exhaustive structured shifts (null_permutations=None)")
    if thai_n:
        notes.append(f"{thai_n} optional Thai site(s) passed and completed")
    else:
        notes.append("no optional Thai site was included; the core China-Japan design is unchanged")
    required_files = [
        writer.figure_dir / "figure01_synthetic_operator_validation.png",
        writer.figure_dir / "figure01b_synthetic_time_local_tracking.png",
        writer.figure_dir / "figure02_data_coverage.png",
        writer.figure_dir / "figure03_forecast_skill_forest.png",
        writer.figure_dir / "figure04_forward_inverse_bridge.png",
        writer.figure_dir / "figure05_cross_region_vector_aime.png",
        writer.figure_dir / "figure06_rolling_atmospheric_alignment.png",
        writer.figure_dir / "figure07_meteorology_ablation.png",
        writer.csv_dir / "table08_forecast_skill.csv",
        writer.csv_dir / "table08b_smap_vs_tuned_ridge.csv",
        writer.csv_dir / "table09b_time_local_oos_inverse.csv",
        writer.csv_dir / "table09c_multihorizon_inverse_ablation.csv",
        writer.csv_dir / "table09d_quadratic_inverse_selection.csv",
        writer.csv_dir / "table01b_source_license_audit.csv",
        writer.csv_dir / "table10_operator_bridge.csv",
        writer.csv_dir / "rolling_vector_aime_operators.csv",
        writer.csv_dir / "table18_meteorology_ablation.csv",
        writer.csv_dir / "table19_tsukuba_pm25_policy_sensitivity.csv",
    ]
    missing = [path.name for path in required_files if not path.exists() or path.stat().st_size == 0]
    if missing:
        problems.append("missing artifacts: " + ", ".join(missing))
    for tex_path in writer.tex_dir.glob("*.tex"):
        text = tex_path.read_text(encoding="utf-8", errors="ignore").lower()
        if " nan " in text or " inf " in text:
            problems.append(f"unsafe literal found in {tex_path.name}")
    return pd.DataFrame(
        [
            {
                "notebook_version": "v9.2_multisite_oos_inverse_operator",
                "package_version": "0.3.1",
                "run_id": run_id,
                "run_mode": config.run_mode,
                "status": "PASS" if not problems else "FAIL",
                "required_beijing_sites_completed": beijing_n,
                "required_tsukuba_sites_completed": tsukuba_n,
                "optional_thai_sites_completed": thai_n,
                "theta_upper_boundary_sites": upper_boundary_n,
                "forecast_ridge_upper_boundary_sites": forecast_ridge_upper_n,
                "quadratic_ridge_upper_boundary_sites": quadratic_ridge_upper_n,
                "scalar_equivalence_max_difference": scalar_difference,
                "problems": "; ".join(problems),
                "notes": "; ".join(notes),
            }
        ]
    )


def _notebook_code_sha256(path: Path) -> str:
    payload = json.loads(path.read_text(encoding="utf-8"))
    code = "\n\n".join(
        "".join(cell.get("source", []))
        for cell in payload.get("cells", [])
        if cell.get("cell_type") == "code"
    )
    return hashlib.sha256(code.encode("utf-8")).hexdigest()


def _source_tree_sha256(root: Path) -> str:
    digest = hashlib.sha256()
    for path in sorted(root.rglob("*.py")):
        digest.update(str(path.relative_to(root)).encode("utf-8"))
        digest.update(path.read_bytes())
    return digest.hexdigest()


def _git_worktree_provenance(package_root: Path | None) -> dict[str, object]:
    if package_root is None:
        return {"git_codev9_tracked": "unavailable", "git_codev9_dirty": "unavailable"}
    try:
        tracked = subprocess.run(
            ["git", "ls-files", "--", "src/tsaime", "notebooks/tsAIME_APR_all_experiments_v9_package.ipynb"],
            cwd=package_root, check=True, capture_output=True, text=True,
        ).stdout.strip()
        dirty = subprocess.run(
            ["git", "status", "--porcelain", "--", "."],
            cwd=package_root, check=True, capture_output=True, text=True,
        ).stdout.strip()
        return {
            "git_codev9_tracked": bool(tracked),
            "git_codev9_dirty": bool(dirty),
            "git_codev9_status": dirty,
        }
    except Exception as exc:
        return {
            "git_codev9_tracked": "unavailable",
            "git_codev9_dirty": "unavailable",
            "git_codev9_status": repr(exc),
        }


def run_apr_pm25(
    base_dir: str | Path,
    config: APRExperimentConfig | None = None,
) -> APRRunResult:
    """Run data audit, synthetic validation, real experiments, and packaging.

    Raw public downloads are cached under ``results/v9_apr/data``.  Generated
    outputs are recreated under ``results/v9_apr/outputs`` on every run.
    """

    config = (config or APRExperimentConfig()).resolved()
    base = Path(base_dir).expanduser().resolve()
    run_root = base / "results" / "v9_apr"
    os.environ.setdefault("MPLBACKEND", "Agg")
    os.environ.setdefault("MPLCONFIGDIR", str(run_root / ".matplotlib"))
    import matplotlib as mpl
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
    output_dir = run_root / "outputs"
    writer = ArtifactWriter(output_dir, clear_existing=config.clear_outputs)
    repository = PublicDataRepository(run_root / "data")
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    notebook_path = (PACKAGE_ROOT / "notebooks" / "tsAIME_APR_all_experiments_v9_package.ipynb") if PACKAGE_ROOT else None
    try:
        git_commit = subprocess.run(
            ["git", "rev-parse", "HEAD"], cwd=PACKAGE_ROOT, check=True,
            capture_output=True, text=True,
        ).stdout.strip()
    except Exception:
        git_commit = "unavailable"
    run_info = {
        "workflow": "APR PM2.5 v9.2",
        "package_version": "0.3.1",
        "run_id": run_id,
        "config": asdict(config),
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": importlib.metadata.version("scipy"),
        "scikit_learn": importlib.metadata.version("scikit-learn"),
        "matplotlib": importlib.metadata.version("matplotlib"),
        "pyEDM": importlib.metadata.version("pyEDM"),
        "git_commit": git_commit,
        "notebook_file_sha256": _sha256(notebook_path) if notebook_path and notebook_path.exists() else "unavailable",
        "notebook_code_sha256": _notebook_code_sha256(notebook_path) if notebook_path and notebook_path.exists() else "unavailable",
        "package_source_sha256": _source_tree_sha256(PACKAGE_ROOT / "src" / "tsaime") if PACKAGE_ROOT else "unavailable",
        **_git_worktree_provenance(PACKAGE_ROOT),
    }
    (writer.log_dir / f"run_info_{run_id}.json").write_text(
        json.dumps(run_info, indent=2, default=str), encoding="utf-8"
    )
    sources = source_table()
    source_latex = sources.loc[sources["dataset"].isin(["Beijing", "Tsukuba"]), ["dataset", "doi", "version", "accessed_on"]].copy()
    source_latex.loc[source_latex["dataset"] == "Beijing", "version"] = "UCI 501 snapshot"
    source_latex["license_status"] = source_latex["dataset"].map(
        {"Beijing": "CC BY 4.0", "Tsukuba": "conflicting notices; recheck", "Thailand": "verify if enabled"}
    )
    writer.save_table(
        sources,
        "table01_data_sources",
        "Data sources and their prespecified analytical roles.",
        "tab:data_sources",
        latex_frame=source_latex,
    )

    synthetic = _synthetic_experiment(config, writer)
    site_frames: dict[tuple[str, str], pd.DataFrame] = {}
    load_failures: list[dict[str, str]] = []
    for site, frame in repository.load_beijing_sites(int(config.beijing_site_limit)).items():
        site_frames[("Beijing", site)] = frame
    try:
        site_frames[("Tsukuba", "NIES_Tsukuba")] = repository.load_tsukuba()
    except Exception as exc:
        load_failures.append(
            {"dataset": "Tsukuba", "site": "NIES_Tsukuba", "error": repr(exc)}
        )
    if config.enable_thailand:
        years = tuple(range(2021, 2026)) if config.run_mode == "publication" else (2024, 2025)
        for site, metadata in THAI_SITES.items():
            try:
                site_frames[("Thailand", site)] = repository.load_thai_site(
                    site, metadata, years, workers=8 if config.run_mode == "publication" else 2
                )
            except Exception as exc:
                load_failures.append({"dataset": "Thailand", "site": site, "error": repr(exc)})

    license_audit = pd.DataFrame(
        repository.license_records,
        columns=["dataset", "file", "landing_page_license", "embedded_file_notice", "accessed_on", "status"],
    )
    writer.save_table(
        license_audit,
        "table01b_source_license_audit",
        "Source-license notices observed in the landing metadata and downloaded files.",
        "tab:source_license_audit",
        latex_frame=license_audit[["dataset", "landing_page_license", "embedded_file_notice", "accessed_on"]].drop_duplicates(),
    )

    audit_tables = []
    plausibility_rows = []
    gate_rows = []
    pair_tables: dict[tuple[str, str], pd.DataFrame] = {}
    target_columns = [f"PM25_future_{horizon}h" for horizon in config.horizons]
    gate = QualityGate(
        minimum_rows=(
            {"train": 1000, "validation": 500, "test": 500}
            if config.run_mode == "publication"
            else {"train": 100, "validation": 50, "test": 50}
        ),
        minimum_coverage=0.60,
    )
    audit_columns = ["PM25", "TEMP", "RH", "PRESS", "RAIN", "WIND_U", "WIND_V"]
    for (dataset, site), frame in site_frames.items():
        plausibility_rows.append(pm25_plausibility_audit(frame, dataset=dataset, site=site))
        analysis_frame = apply_pm25_policy(frame, config.main_pm25_policy)
        study_start, _, _, study_end = SPLITS[dataset].timestamps()
        audit_tables.append(
            audit_hourly_site(
                analysis_frame, audit_columns, dataset=dataset, site=site,
                start=study_start, end=study_end,
            )
        )
        candidates, complete, states, targets = prepare_pm25_multihorizon_pairs(
            analysis_frame,
            config.horizons,
            SPLITS[dataset],
            origin_step_hours=int(config.origin_step_hours),
            required_states=STATE_COLUMNS,
        )
        if list(targets) != target_columns or tuple(states) != STATE_COLUMNS:
            raise RuntimeError("workflow state or target schema changed unexpectedly")
        gate_row = evaluate_quality_gate(
            candidates, complete, gate, dataset=dataset, site=site
        )
        gate_rows.append(gate_row)
        if gate_row["quality_gate"] == "PASS":
            pair_tables[(dataset, site)] = complete

    audit = pd.concat(audit_tables, ignore_index=True) if audit_tables else pd.DataFrame()
    quality_gate = pd.DataFrame(gate_rows)
    plausibility = pd.DataFrame(plausibility_rows)
    plausibility["meteorology_flag_n"] = plausibility[[
        "rh_outside_0_100_n", "pressure_outside_800_1100_n", "negative_rain_n"
    ]].sum(axis=1)
    failure_table = pd.DataFrame(load_failures, columns=["dataset", "site", "error"])
    writer.save_table(
        audit,
        "table05_data_audit_by_site_year",
        "Hourly data completeness and maximum gaps by site and year.",
        "tab:data_audit",
        latex_frame=audit.groupby(["dataset", "year"], as_index=False).agg(
            sites=("site", "nunique"), median_complete_coverage=("all_required_coverage", "median"),
            minimum_complete_coverage=("all_required_coverage", "min"),
        ),
    )
    writer.save_table(
        plausibility,
        "table05b_pm25_plausibility_audit",
        "PM2.5 ranges and negative observations before policy application.",
        "tab:pm25_plausibility",
        latex_frame=plausibility.groupby("dataset", as_index=False).agg(
            sites=("site", "nunique"), pm25_negative_n=("pm25_negative_n", "sum"),
            pm25_min=("pm25_min", "min"), pm25_max=("pm25_max", "max"),
            meteorology_flag_n=("meteorology_flag_n", "sum"),
        ),
    )
    writer.save_table(
        quality_gate,
        "table06_quality_gate",
        "Prespecified source-target pair quality gate.",
        "tab:quality_gate",
        latex_frame=quality_gate.groupby("dataset", as_index=False).agg(
            sites=("site", "nunique"), passed=("quality_gate", lambda values: int((values == "PASS").sum())),
            train_min=("train_coverage", "min"),
            validation_min=("validation_coverage", "min"),
            test_min=("test_coverage", "min"),
        ),
    )
    writer.save_table(
        failure_table,
        "table07_load_failures",
        "Data sources or site experiments that could not be completed.",
        "tab:load_failures",
    )
    _figure_data_coverage(audit, writer)

    real_results: dict[tuple[str, str], dict[str, Any]] = {}
    for (dataset, site), pairs in pair_tables.items():
        try:
            real_results[(dataset, site)] = _run_real_site(
                dataset, site, pairs, target_columns, config
            )
        except Exception as exc:
            load_failures.append({"dataset": dataset, "site": site, "error": repr(exc)})
    failure_table = pd.DataFrame(load_failures, columns=["dataset", "site", "error"])
    forecast = _concatenate(real_results, "forecast")
    forecast_comparison = _concatenate(real_results, "forecast_comparison")
    forecast_ridge_selection = _concatenate(real_results, "forecast_ridge_selection")
    inverse = _concatenate(real_results, "inverse")
    quadratic_selection = _concatenate(real_results, "quadratic_selection")
    multihorizon_inverse = _concatenate(real_results, "multihorizon_inverse")
    rolling_inverse_oos = _concatenate(real_results, "rolling_inverse_oos")
    rolling_inverse_windows = _concatenate(real_results, "rolling_inverse_windows")
    diagnostics = _concatenate(real_results, "diagnostics")
    operators = _concatenate(real_results, "operators")
    nulls = _concatenate(real_results, "null")
    theta = _concatenate(real_results, "theta")
    ridge = _concatenate(real_results, "ridge_selection")
    ablation_parts = []
    for key, result in real_results.items():
        ablation_parts.append(
            _meteorology_ablation(
                key[0], key[1], pair_tables[key], target_columns,
                result["test_predictions"], config,
            )
        )
    ablation = pd.concat(ablation_parts, ignore_index=True) if ablation_parts else pd.DataFrame()
    tsukuba_key = ("Tsukuba", "NIES_Tsukuba")
    pm25_policy = (
        _pm25_policy_sensitivity(
            site_frames[tsukuba_key], SPLITS["Tsukuba"], config,
            dataset="Tsukuba", site="NIES_Tsukuba",
        )
        if tsukuba_key in site_frames else pd.DataFrame()
    )
    writer.save_table(
        forecast,
        "table08_forecast_skill",
        "Strict out-of-sample forecast performance and block-bootstrap skill intervals.",
        "tab:forecast_skill",
        latex_frame=_compact_forecast_table(forecast),
    )
    writer.save_table(
        forecast_comparison,
        "table08b_smap_vs_tuned_ridge",
        "Paired strict OOS comparison of S-Map with validation-tuned Ridge forecasts.",
        "tab:smap_ridge",
        latex_frame=forecast_comparison.groupby(["dataset", "horizon_h"], as_index=False).agg(
            sites=("site", "nunique"), median_skill=("skill_SMap_vs_tuned_ridge", "median"),
            positive_CI_sites=("skill_CI_low", lambda values: int((values > 0).sum())),
        ),
    )
    writer.save_table(
        forecast_ridge_selection,
        "table08c_ridge_forecast_selection",
        "Validation-only regularization selection for the linear Ridge forecast baseline.",
        "tab:ridge_forecast_selection",
        latex_frame=forecast_ridge_selection.loc[forecast_ridge_selection["selected"], ["dataset", "site", "alpha", "validation_standardized_RMSE"]],
    )
    writer.save_table(
        inverse,
        "table09_inverse_reconstruction",
        "Out-of-sample reconstruction of standardized state variables from forecast vectors.",
        "tab:inverse_reconstruction",
        latex_frame=_compact_inverse_table(inverse),
    )
    writer.save_table(
        rolling_inverse_oos,
        "table09b_time_local_oos_inverse",
        "Past-window to future-block inverse reconstruction with moving-block intervals.",
        "tab:rolling_inverse_oos",
        latex_frame=rolling_inverse_oos.groupby(["dataset", "feature"], as_index=False).agg(
            sites=("site", "nunique"), median_skill=("skill_AIME_vs_marginal", "median"),
            positive_CI_sites=("skill_CI_low", lambda values: int((values > 0).sum())),
        ),
    )
    writer.save_csv(rolling_inverse_windows, "rolling_oos_inverse_windows.csv")
    writer.save_table(
        multihorizon_inverse,
        "table09c_multihorizon_inverse_ablation",
        "Inverse reconstruction using single-horizon and joint multi-horizon forecast outputs.",
        "tab:multihorizon_inverse",
        latex_frame=multihorizon_inverse.groupby(["dataset", "output_set"], as_index=False).agg(
            sites=("site", "nunique"), median_RMSE=("inverse_RMSE", "median"),
            median_gain_vs_best_single=("gain_all_vs_best_single", "median"),
        ),
    )
    writer.save_table(
        quadratic_selection,
        "table09d_quadratic_inverse_selection",
        "Validation-only regularization selection for the quadratic inverse diagnostic.",
        "tab:quadratic_inverse_selection",
        latex_frame=quadratic_selection.loc[quadratic_selection["selected"], ["dataset", "site", "ridge", "validation_inverse_RMSE"]],
    )
    writer.save_table(
        diagnostics,
        "table10_operator_bridge",
        "Rolling forward-inverse operator bridge diagnostics.",
        "tab:operator_bridge",
        latex_frame=_compact_bridge_table(diagnostics),
    )
    writer.save_table(
        nulls,
        "table11_structured_null",
        "Weekly phase-preserving circular-shift test of inverse-operator magnitude.",
        "tab:structured_null",
        latex_frame=nulls.rename(columns={
            "exact_weekly_phase_shift_p_value": "p_exact",
            "null_unique_shifts": "n_shifts",
            "minimum_attainable_p_value": "min_p",
        })[["dataset", "site", "p_exact", "n_shifts", "min_p"]],
    )
    writer.save_table(
        theta,
        "table12_smap_theta_selection",
        "Validation-only S-Map localization selection.",
        "tab:smap_theta",
        latex_frame=_compact_theta_table(theta, max(config.theta_grid)),
    )
    writer.save_table(
        ridge,
        "table13_aime_ridge_selection",
        "Validation-only vector ts-AIME regularization selection.",
        "tab:aime_ridge",
        latex_frame=_compact_ridge_table(ridge),
    )
    writer.save_table(
        failure_table,
        "table07_load_failures",
        "Data sources or site experiments that could not be completed.",
        "tab:load_failures",
    )
    writer.save_csv(operators, "rolling_vector_aime_operators.csv")
    writer.save_csv(pd.DataFrame(repository.records), "download_provenance.csv")
    writer.save_table(
        ablation,
        "table18_meteorology_ablation",
        "Strict OOS forecast comparison of full and pressure/wind-reduced S-Map models.",
        "tab:meteorology_ablation",
        latex_frame=ablation.groupby(["dataset", "reduced_model", "horizon_h"], as_index=False).agg(
            sites=("site", "nunique"), median_skill_full_vs_reduced=("skill_full_vs_reduced", "median"),
            sites_positive_CI=("skill_CI_low", lambda values: int((values > 0).sum())),
        ),
    )
    writer.save_table(
        pm25_policy,
        "table19_tsukuba_pm25_policy_sensitivity",
        "Tsukuba strict OOS sensitivity to negative PM2.5 handling.",
        "tab:pm25_policy",
        latex_frame=pm25_policy.rename(columns={
            "horizon_h": "h", "test_n": "n",
            "skill_vs_persistence": "skill", "skill_CI_low": "CI_low",
            "skill_CI_high": "CI_high",
        })[["policy", "h", "n", "RMSE", "skill", "CI_low", "CI_high"]],
    )
    _publication_figures(forecast, diagnostics, operators, real_results, ablation, config, writer)
    sensitivity = _window_sensitivity(real_results, config)
    writer.save_table(
        sensitivity,
        "table16_window_sensitivity",
        "Sensitivity of vector ts-AIME magnitude to trailing window length.",
        "tab:window_sensitivity",
    )
    readiness = _readiness_report(
        config, run_id, synthetic, real_results, forecast, theta, nulls,
        ablation, pm25_policy, writer,
    )
    readiness_latex = pd.DataFrame(
        {
            "check": ["status", "run mode", "Beijing sites", "Tsukuba sites", "theta upper-bound sites", "scalar identity error"],
            "value": [
                readiness.iloc[0]["status"], readiness.iloc[0]["run_mode"],
                readiness.iloc[0]["required_beijing_sites_completed"],
                readiness.iloc[0]["required_tsukuba_sites_completed"],
                readiness.iloc[0]["theta_upper_boundary_sites"],
                readiness.iloc[0]["scalar_equivalence_max_difference"],
            ],
        }
    )
    writer.save_table(
        readiness,
        "table17_computational_readiness",
        "Automatic computational readiness report; it is not an acceptance guarantee.",
        "tab:readiness",
        latex_frame=readiness_latex,
    )
    writer.write_text(
        "DATA_AVAILABILITY.txt",
        """Data availability

Beijing hourly observations were obtained from the UCI Beijing Multi-Site Air Quality dataset (DOI: 10.24432/C5RK5G; CC BY 4.0; accessed 2026-09-04). Tsukuba hourly observations were obtained from the NIES Atmospheric Environmental Regional Observation System dataset (DOI: 10.17595/20250418.001; Version 1.0; accessed 2026-09-04). The NIES landing page displayed CC BY-NC-ND 4.0 while the downloaded text header displayed CC BY 4.0 at verification; users must follow the publisher's current controlling terms. Raw source data are not redistributed in this repository or result archive. Download URLs, checksums, and cache timestamps are recorded in table01_data_sources.csv and download_provenance.csv.
""",
    )
    writer.write_text(
        "README_RESULTS.txt",
        f"""ts-AIME APR v9.2 package workflow run {run_id}

Computational readiness status: {readiness.iloc[0]['status']}

S-Map estimates a local forward coefficient matrix. Vector-output ts-AIME
estimates a covariance-weighted output-to-input reconstruction operator.

Required data: UCI Beijing Multi-Site and NIES Tsukuba.
Data DOI, version, source-license status, and access date are in table01_data_sources.csv.
The NIES landing page and downloaded file header showed different license notices on 2026-09-04; current publisher terms must be checked before public release.
Optional data: Thailand, only after the prespecified quality gate.

Real-data operators are associative forecast diagnostics. They are not causal
effects and are not source-apportionment estimates.
""",
    )
    writer.manifest()
    zip_path = writer.create_zip(f"tsAIME_APR_v9_2_package_outputs_{run_id}.zip")
    return APRRunResult(
        run_id=run_id,
        run_root=run_root,
        output_dir=output_dir,
        zip_path=zip_path,
        readiness=readiness,
        quality_gate=quality_gate,
        forecast_skill=forecast,
    )


## 7. Confirm the prespecified experiment contract

`publication` runs the complete required study.

Use `smoke` only for a quick installation check.

Thailand is disabled by default and is included only after the same prespecified quality gate.

Set `TSAIME_V9_ENABLE_THAI=1` before starting the kernel to request the optional extension.


In [ ]:
RUN_MODE = os.environ.get("TSAIME_V9_RUN_MODE", "publication").strip().lower()
ENABLE_THAILAND = os.environ.get("TSAIME_V9_ENABLE_THAI", "0") == "1"

CONFIG = APRExperimentConfig(
    run_mode=RUN_MODE,
    enable_thailand=ENABLE_THAILAND,
    seed=20260904,
    horizons=(1, 6, 24),
    inverse_ridge_grid=(0.0, 1e-3, 1e-2, 1e-1, 1.0),
    aime_window_days=28,
    aime_window_sensitivity_days=(14, 28, 56),
)

display(source_table())
display(
    pd.DataFrame(
        [
            {
                "dataset": dataset,
                "start": split.timestamps()[0],
                "train_end": split.timestamps()[1],
                "validation_end": split.timestamps()[2],
                "test_end": split.timestamps()[3],
            }
            for dataset, split in SPLITS.items()
        ]
    )
)
display(pd.DataFrame({"APR_state_variable": STATE_COLUMNS}))
print(CONFIG.resolved())


## 8. Run every experiment and create the manuscript bundle

Future targets are created on a complete hourly grid before incomplete pairs are removed.

Validation data select S-Map localization and inverse regularization.

Test rows never enter those selection steps or the S-Map library.


In [ ]:
RESULT = run_apr_pm25(RESULT_BASE_DIR, CONFIG)

display(RESULT.readiness)
display(RESULT.quality_gate)
print("Figures and tables:", RESULT.output_dir)
print("ZIP:", RESULT.zip_path)

if RESULT.readiness.iloc[0]["status"] != "PASS":
    print(
        "The diagnostic ZIP was created, but the computational contract did not pass. "
        "Read table17_computational_readiness.csv before manuscript use."
    )


## 9. Result summary for manuscript planning

Positive confidence-interval lower bounds indicate improvement over persistence under the specified moving-block bootstrap.


In [ ]:
FORECAST = RESULT.forecast_skill
SMAP_SUMMARY = (
    FORECAST.loc[FORECAST["model"] == "SMap"]
    .groupby("horizon_h", as_index=False)
    .agg(
        completed_sites=("site", "nunique"),
        median_skill_vs_persistence=("skill_vs_persistence", "median"),
        sites_with_positive_CI=("skill_CI_low", lambda values: int((values > 0).sum())),
        median_rho=("rho", "median"),
    )
)
display(SMAP_SUMMARY)
